# Hotel Booking Cancellation Project: All Code in One Notebook

This notebook was generated from the current project source so the whole implementation is available in one place.

Preprocessing notes:
- the active app uses the reservation dataset pipeline aligned to the reference notebook
- reservation preprocessing keeps the compact notebook-style feature set, applies SMOTE on the training split, one-hot encodes categorical fields with `drop_first=True`, and scales the nightly rate column
- numeric features outside the reservation benchmark still use the shared `StandardScaler` pipeline
- Random Forest now uses `RandomizedSearchCV` to search stronger hyperparameters such as `n_estimators`, `max_depth`, and `max_features`
- class balancing is applied across the model set through native class weights, balanced sample weights, oversampling wrappers, and class-weighted deep learning fits
- the deployed Streamlit app now exposes only the notebook-aligned reservation benchmark path and its matching model artifacts
- UI field labels were renamed for presentation, but the underlying saved model schema remains synchronized with the benchmark artifacts

Where key answers live:
- core model classes: `hotel_app/ml/models/`
- sigmoid / probability helper for non-probability estimators: `hotel_app/ml/data.py::_positive_probabilities`
- TensorFlow sigmoid output for ANN/RNN/LSTM: `hotel_app/ml/deep.py`
- train/test metric generation: `hotel_app/ml/testing.py` and `hotel_app/ml/training.py`
- saved metric tables: `artifacts/reports/holdout_summary.csv`

Contents:
- reporting and terminal scripts
- Streamlit dashboard
- service layer
- machine learning pipeline
- all model classes

Each section below shows the source of one project file.


## Model Reference Map

This table is meant to answer the common doctor / examiner questions about where each model lives, how it is balanced, where its probability output comes from, and where the training metrics are saved.

| Model | Class File | Estimator / Search | Balancing | Probability / Sigmoid Path | Metrics Saved |
| --- | --- | --- | --- | --- | --- |
| Logistic Regression | `hotel_app/ml/models/logistic.py` | LogisticRegression inside GridSearchCV | class_weight='balanced' | native logistic sigmoid via predict_proba | saved in holdout_summary.csv as train_* columns |
| KNN | `hotel_app/ml/models/knn.py` | KNeighborsClassifier inside GridSearchCV | BalancedClassifierWrapper with oversampling | native neighbor vote probabilities | saved in holdout_summary.csv as train_* columns |
| Decision Tree | `hotel_app/ml/models/decision_tree.py` | DecisionTreeClassifier inside GridSearchCV | class_weight='balanced' | native tree leaf probabilities | saved in holdout_summary.csv as train_* columns |
| Random Forest | `hotel_app/ml/models/random_forest.py` | RandomForestClassifier inside RandomizedSearchCV | class_weight='balanced_subsample' | average positive-class probability over trees | saved in holdout_summary.csv as train_* columns |
| XGBoost | `hotel_app/ml/models/xgboost_model.py` | XGBClassifier with tuned tree parameters | boosted tree regularization and weighted loss behavior | native predict_proba from XGBoost | saved in holdout_summary.csv as train_* columns |
| SVM | `hotel_app/ml/models/svm.py` | RBF-kernel SVC inside a stratified subsampling wrapper and GridSearchCV | class_weight='balanced' | native SVC predict_proba | saved in holdout_summary.csv as train_* columns |
| ANN | `hotel_app/ml/models/ann.py` | TensorFlow KerasTabularClassifier or MLP fallback | class weights in TensorFlow, oversampling in fallback | final sigmoid output layer in hotel_app/ml/deep.py | saved in holdout_summary.csv as train_* columns |
| RNN | `hotel_app/ml/models/rnn.py` | TensorFlow KerasTabularClassifier in rnn mode | class-weighted TensorFlow fit | final sigmoid output layer in hotel_app/ml/deep.py | saved in holdout_summary.csv as train_* columns |
| LSTM | `hotel_app/ml/models/lstm.py` | TensorFlow KerasTabularClassifier in lstm mode | class-weighted TensorFlow fit | final sigmoid output layer in hotel_app/ml/deep.py | saved in holdout_summary.csv as train_* columns |

## `build_pdf_report.py`


In [ ]:
from hotel_app import BenchmarkPdfBuilder


if __name__ == "__main__":
    path = BenchmarkPdfBuilder("artifacts").build()
    print(path)


## `build_word_report.py`


In [ ]:
import argparse
from pathlib import Path
import pandas as pd
from docx import Document
from docx.shared import Pt, Inches, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml.ns import qn

from hotel_app.ml.models import MODEL_REGISTRY

def add_heading(doc, text, level=1):
    heading = doc.add_heading(text, level=level)
    for run in heading.runs:
        run.font.name = 'Arial'
        if level == 1:
            run.font.color.rgb = RGBColor(15, 61, 86) # Dark teal
        elif level == 2:
            run.font.color.rgb = RGBColor(30, 100, 130)

def add_paragraph(doc, text, bold=False, italic=False, align=WD_ALIGN_PARAGRAPH.LEFT, style=None):
    p = doc.add_paragraph(style=style)
    p.alignment = align
    run = p.add_run(text)
    run.font.name = 'Arial'
    run.font.size = Pt(11)
    run.bold = bold
    run.italic = italic
    return p

def add_image(doc, image_path, width_inches=6.0, caption=""):
    if Path(image_path).exists():
        doc.add_picture(image_path, width=Inches(width_inches))
        if caption:
            p = add_paragraph(doc, caption, italic=True, align=WD_ALIGN_PARAGRAPH.CENTER)
            p.runs[0].font.size = Pt(9)
            p.runs[0].font.color.rgb = RGBColor(100, 100, 100)
            doc.add_paragraph("\n")

def add_code_block(doc, code):
    # Create a light gray background paragraph for code if possible, or just a monospaced block
    p = doc.add_paragraph()
    run = p.add_run(code)
    run.font.name = 'Courier New'
    run.font.size = Pt(9)
    run.font.color.rgb = RGBColor(40, 40, 40)
    p.paragraph_format.left_indent = Inches(0.2)
    doc.add_paragraph("\n")

def build_report(output_file="Hotel_Cancellation_Final_Report.docx", team_members=None):
    if team_members is None:
        team_members = ["[Insert Team Member 1 Name]", "[Insert Team Member 2 Name]", "[Insert Team Member 3 Name]"]

    doc = Document()

    # --- Title Page ---
    title = doc.add_heading("Hotel Booking Cancellation Prediction", 0)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    subtitle = doc.add_paragraph("Machine Learning Final Project Report")
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    subtitle.runs[0].font.size = Pt(14)
    subtitle.runs[0].bold = True
    doc.add_paragraph("\n")

    add_image(doc, "artifacts/plots/guest_segmentation.png", width_inches=5.5, caption="Guest Segmentation Visualization (PCA & K-Means)")
    
    doc.add_paragraph("\n\n")
    team_heading = doc.add_heading("Team Members", level=2)
    team_heading.alignment = WD_ALIGN_PARAGRAPH.CENTER
    for member in team_members:
        p = add_paragraph(doc, member, align=WD_ALIGN_PARAGRAPH.CENTER)
        p.runs[0].font.size = Pt(12)

    doc.add_page_break()

    # --- Section 1: Approach & Architecture ---
    add_heading(doc, "1. Project Overview & Methodology", level=1)
    add_paragraph(doc, "The objective of this project is to build a robust machine learning pipeline to predict hotel booking cancellations. We adopted a terminal-first workflow that separates model training and evaluation from the user-facing deployment (Streamlit app). This ensures a clean, production-ready architecture where evaluation metrics are completely honest.")
    
    add_heading(doc, "Data Processing & Engineering", level=2)
    add_paragraph(doc, "We engineered several high-signal features from the raw dataset:")
    doc.add_paragraph("• total_nights: Combination of weekend and weekday nights.", style='List Bullet')
    doc.add_paragraph("• family_booking: Indicates if the booking includes children or babies.", style='List Bullet')
    doc.add_paragraph("• previous_cancel_rate: The historical cancellation rate of the guest.", style='List Bullet')
    doc.add_paragraph("• room_match: Flag indicating if the assigned room matches the reserved room.", style='List Bullet')
    add_paragraph(doc, "Crucially, leakage columns like 'reservation_status' and 'reservation_status_date' were aggressively removed to ensure models generalize to real-world deployment scenarios where this data is unavailable at the time of booking.")

    add_heading(doc, "Pipeline Implementation Details", level=2)
    add_paragraph(doc, "Below is an excerpt of our `ModelTrainer` class in `hotel_app/ml/training.py`. It showcases the robust scikit-learn pipeline construction we used to ensure the preprocessor is bound tightly to the estimator, preventing data leakage during cross-validation.")
    code_snippet = '''def train_model(self, model_spec: BaseHotelModel, x_train: pd.DataFrame, y_train: pd.Series) -> Pipeline:
    # Build preprocessor (Scaling, One-Hot Encoding, Imputation) based on training data
    preprocessor = self.processor.build_preprocessor(x_train)
    
    # Construct a Sklearn Pipeline
    pipeline = model_spec.build_pipeline(preprocessor)
    
    # Fit the entire pipeline
    return pipeline.fit(x_train, y_train)'''
    add_code_block(doc, code_snippet)

    # --- Section 2: Models Trained ---
    doc.add_page_break()
    add_heading(doc, "2. Models Evaluated & Source Code", level=1)
    add_paragraph(doc, "We evaluated an exhaustive suite of 14 distinct machine learning algorithms. Each model is encapsulated in its own class, inheriting from `BaseHotelModel`. Below is the exact implementation code and functions used for each model in our pipeline.")
    doc.add_paragraph("\n")

    import inspect
    for name, model_class in MODEL_REGISTRY.items():
        doc.add_heading(name, level=2)
        try:
            source_code = inspect.getsource(model_class)
            add_code_block(doc, source_code.strip())
        except Exception as e:
            add_paragraph(doc, f"[Could not extract source code: {e}]", italic=True)
        doc.add_paragraph("\n")

    # --- Section 3: Metrics ---
    doc.add_page_break()
    add_heading(doc, "3. Evaluation Metrics (Holdout Split)", level=1)
    add_paragraph(doc, "Models were evaluated using an honest 30% holdout split. To provide complete transparency, the training time metrics below reflect both the benchmark phase and the full-dataset retraining phase required for deployment. The table below imports the live CSV metrics produced during the terminal run.")
    
    metrics_path = Path("artifacts/reports/holdout_summary.csv")
    if metrics_path.exists():
        df = pd.read_csv(metrics_path)
        # Select key columns
        df = df[['model', 'accuracy', 'f1', 'roc_auc', 'training_time_sec']].round(4)
        
        table = doc.add_table(rows=1, cols=5)
        table.style = 'Table Grid'
        hdr_cells = table.rows[0].cells
        hdr_cells[0].text = 'Model'
        hdr_cells[1].text = 'Accuracy'
        hdr_cells[2].text = 'F1 Score'
        hdr_cells[3].text = 'ROC-AUC'
        hdr_cells[4].text = 'Training Time (s)'
        
        for index, row in df.iterrows():
            row_cells = table.add_row().cells
            row_cells[0].text = str(row['model'])
            row_cells[1].text = str(row['accuracy'])
            row_cells[2].text = str(row['f1'])
            row_cells[3].text = str(row['roc_auc'])
            row_cells[4].text = str(row['training_time_sec'])
        doc.add_paragraph("\n")
    else:
        add_paragraph(doc, "[Metrics CSV not found. Please run training first.]", italic=True)

    add_heading(doc, "Performance & Timing Graphs", level=2)
    add_image(doc, "artifacts/plots/cross_validation_f1.png", width_inches=6.0, caption="Cross-Validation F1 Scores Across 5 Folds")
    add_image(doc, "artifacts/plots/timing_metrics.png", width_inches=6.0, caption="Model Training & Inference Time Analysis")

    # --- Section 4: Confusion Matrices ---
    doc.add_page_break()
    add_heading(doc, "4. Precision vs. Recall (Confusion Matrices)", level=1)
    add_paragraph(doc, "Below are the confusion matrices for several key models, illustrating the false-positive vs false-negative trade-offs made during prediction.")
    
    add_image(doc, "artifacts/plots/random_forest_confusion_matrix.png", width_inches=5.0, caption="Random Forest Confusion Matrix")
    add_image(doc, "artifacts/plots/xgboost_confusion_matrix.png", width_inches=5.0, caption="XGBoost Confusion Matrix")
    add_image(doc, "artifacts/plots/svm_confusion_matrix.png", width_inches=5.0, caption="SVM Confusion Matrix")

    # --- Section 5: Interpretability ---
    doc.add_page_break()
    add_heading(doc, "5. Explainability (SHAP Analysis)", level=1)
    add_paragraph(doc, "Machine Learning models should not act as black boxes. To provide transparency to the hotel management team, we implemented SHAP (SHapley Additive exPlanations) to explain the global decision-making process of the best performing models.")
    
    add_image(doc, "artifacts/plots/random_forest_shap_summary.png", width_inches=6.0, caption="Global SHAP Summary Plot: Features ranked by their impact on Cancellation Probability")
    
    add_paragraph(doc, "The SHAP plot visually demonstrates how features like 'deposit_type' (Non-Refundable deposits massively reduce cancellation risk) and 'lead_time' (longer lead times drastically increase cancellation risk) drive the predictions.")

    # Save Document
    doc.save(output_file)
    print(f"Detailed report successfully saved to {output_file}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Generate Detailed Word Report")
    parser.add_argument("--members", nargs='+', help="List of team member names", default=None)
    args = parser.parse_args()
    
    build_report(team_members=args.members)


## `train_terminal.py`


In [ ]:
from __future__ import annotations

import argparse
from pathlib import Path

import pandas as pd

from hotel_app.ml import TerminalTrainingRunner


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(
        description="Train hotel cancellation models from the terminal and save artifacts."
    )
    parser.add_argument("--data", default="hotel_bookings.csv", help="Path to the dataset CSV file.")
    parser.add_argument("--output", default="artifacts", help="Directory for saved models and reports.")
    parser.add_argument("--cv-folds", type=int, default=5, help="Number of stratified CV folds. Use 0 or 1 to skip CV.")
    parser.add_argument("--ann-epochs", type=int, default=250, help="ANN max iterations.")
    parser.add_argument("--rnn-epochs", type=int, default=10, help="RNN epochs when TensorFlow is available.")
    parser.add_argument("--lstm-epochs", type=int, default=10, help="LSTM epochs when TensorFlow is available.")
    parser.add_argument("--shap-rows", type=int, default=250, help="Rows to use for SHAP plots.")
    parser.add_argument(
        "--feature-preset",
        choices=["honest", "high_score"],
        default=None,
        help="Feature preset to use for training artifacts.",
    )
    parser.add_argument(
        "--keep-leakage-features",
        action="store_true",
        help="Keep leakage features such as reservation_status in training and prediction artifacts.",
    )
    parser.add_argument(
        "--models",
        nargs="+",
        default=None,
        help="Optional list of model names to train, for example --models \"Logistic Regression\" KNN \"Random Forest\" XGBoost ANN",
    )
    return parser


def main() -> None:
    args = build_parser().parse_args()
    runner = TerminalTrainingRunner()
    results = runner.run(
        data_path=args.data,
        output_dir=args.output,
        cv_folds=args.cv_folds,
        ann_epochs=args.ann_epochs,
        rnn_epochs=args.rnn_epochs,
        lstm_epochs=args.lstm_epochs,
        shap_rows=args.shap_rows,
        selected_models=args.models,
        remove_leakage_features=not args.keep_leakage_features,
        feature_preset=args.feature_preset,
    )

    holdout = results["holdout_summary"].copy()
    cross_validation = results["cross_validation_results"].copy()
    metadata = results["metadata"]

    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 220)

    print("\nTraining complete.\n")
    test_ratio = float(metadata.get("test_ratio", 0.2))
    print(f"Holdout summary ({int(round(test_ratio * 100))}% test split plus benchmark-train metrics):")
    print(holdout.round(4).to_string(index=False))

    print("\n5-fold cross-validation means:")
    if not cross_validation.empty and "fold" in cross_validation.columns:
        cv_means = cross_validation[cross_validation["fold"].astype(str) == "mean"].copy()
    else:
        cv_means = pd.DataFrame()
    if not cv_means.empty:
        print(cv_means.round(4).to_string(index=False))
    else:
        print("No cross-validation rows were produced.")

    print("\nSaved artifacts:")
    print(f"- Output directory: {Path(args.output).resolve()}")
    print(f"- Best model: {metadata.get('best_model')}")
    if metadata.get("skipped_models"):
        print("- Skipped models:")
        for name, reason in metadata["skipped_models"].items():
            print(f"  {name}: {reason}")


if __name__ == "__main__":
    main()


## `streamlit_app.py`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import time
from typing import Any, Dict, List

import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st
from plotly.subplots import make_subplots

from hotel_app.ml import HotelDataProcessor, SHAPAnalyzer, _positive_probabilities


ARTIFACTS_DIR = Path("artifacts")
DATA_PATH = (
    Path("hotel reservation data set .csv")
    if Path("hotel reservation data set .csv").exists()
    else Path("hotel_booking.csv")
    if Path("hotel_booking.csv").exists()
    else Path("hotel_bookings.csv")
)
EXCLUDED_FEATURES = {"arrival_date_year"}
FIELD_LABELS = {
    "type_of_meal": "Meal Package",
    "car_parking_space": "Parking Need Flag",
    "room_type": "Room Category",
    "lead_time": "Advance Booking Window",
    "market_segment_type": "Booking Channel",
    "average_price": "Nightly Room Rate",
    "special_requests": "Special Request Count",
    "number_of_children_and_adults": "Traveler Count",
    "number_of_total_nights": "Stay Length Band",
    "day_name": "Reservation Weekday",
    "month": "Reservation Month",
    "year": "Reservation Year",
    "cancellation_ratio": "Past Cancellation Share",
    "first_time_visitor": "New Guest Indicator",
}
FIELD_OPTION_LABELS = {
    "car_parking_space": {0: "No Parking Needed", 1: "Parking Needed"},
    "lead_time": {
        0: "Same Day",
        1: "Short Notice",
        2: "Medium Term",
        3: "Long Term",
        4: "Very Long Term",
    },
    "number_of_total_nights": {
        0: "Day Use",
        1: "Short Stay",
        2: "Week Stay",
        3: "Two Weeks Stay",
        4: "Long Stay",
    },
    "day_name": {
        0: "Monday",
        1: "Tuesday",
        2: "Wednesday",
        3: "Thursday",
        4: "Friday",
        5: "Saturday",
        6: "Sunday",
    },
    "first_time_visitor": {0: "Returning Guest", 1: "First-Time Guest"},
}
MODEL_METRIC_COLUMNS = [
    ("train_accuracy", "train_accuracy"),
    ("accuracy", "test_accuracy"),
    ("train_precision", "train_precision"),
    ("precision", "test_precision"),
    ("train_recall", "train_recall"),
    ("recall", "test_recall"),
    ("train_f1", "train_f1"),
    ("f1", "test_f1"),
    ("train_balanced_accuracy", "train_balanced_accuracy"),
    ("balanced_accuracy", "test_balanced_accuracy"),
    ("train_roc_auc", "train_roc_auc"),
    ("roc_auc", "test_roc_auc"),
    ("train_average_precision", "train_average_precision"),
    ("average_precision", "test_average_precision"),
    ("train_brier_score", "train_brier_score"),
    ("brier_score", "test_brier_score"),
    ("train_log_loss", "train_log_loss"),
    ("log_loss", "test_log_loss"),
    ("train_mcc", "train_mcc"),
    ("mcc", "test_mcc"),
]


def _format_duration(seconds: Any) -> str:
    try:
        value = float(seconds)
    except (TypeError, ValueError):
        return "N/A"
    if value < 60:
        return f"{value:.1f}s"
    minutes, remainder = divmod(int(round(value)), 60)
    if minutes < 60:
        return f"{minutes}m {remainder}s"
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h {minutes}m"


class DashboardStyle:
    @staticmethod
    def apply() -> None:
        st.markdown(
            """
            <style>
            @import url('https://fonts.googleapis.com/css2?family=Manrope:wght@400;500;600;700;800&display=swap');

            html, body, [class*="css"] {
                font-family: 'Manrope', sans-serif;
            }

            .stApp {
                background:
                    radial-gradient(circle at 8% 10%, rgba(14, 165, 233, 0.10), transparent 24%),
                    radial-gradient(circle at 92% 12%, rgba(245, 158, 11, 0.12), transparent 26%),
                    radial-gradient(circle at 50% 100%, rgba(13, 148, 136, 0.10), transparent 36%),
                    linear-gradient(180deg, #f8fafc 0%, #edf7f5 100%);
            }

            .block-container {
                padding-top: 1.4rem;
                padding-bottom: 2rem;
            }

            .hero-shell {
                position: relative;
                overflow: hidden;
                padding: 40px 38px 36px;
                border-radius: 30px;
                margin-bottom: 22px;
                color: white;
                background: linear-gradient(-45deg, #07111f, #0f3d56, #0f766e, #064e3b);
                background-size: 400% 400%;
                box-shadow: 0 34px 90px rgba(7, 17, 31, 0.20);
                animation: fadeLift .8s ease both, heroGradientWave 12s ease infinite;
            }

            .hero-shell::before,
            .hero-shell::after {
                content: "";
                position: absolute;
                left: -10%;
                width: 120%;
                height: 300px;
                border-radius: 42%;
                opacity: .12;
                pointer-events: none;
            }

            .hero-shell::before {
                bottom: -220px;
                background: linear-gradient(90deg, #38bdf8, #0ea5e9, #34d399);
                animation: waveDriftA 14s linear infinite;
            }

            .hero-shell::after {
                bottom: -240px;
                background: linear-gradient(90deg, #fde68a, #f59e0b, #fb7185);
                animation: waveDriftB 18s linear infinite;
            }

            @keyframes heroGradientWave {
                0% { background-position: 0% 50%; }
                50% { background-position: 100% 50%; }
                100% { background-position: 0% 50%; }
            }

            .hero-topline {
                position: relative;
                z-index: 2;
                display: inline-flex;
                align-items: center;
                gap: 10px;
                padding: 8px 14px;
                border-radius: 999px;
                background: rgba(255,255,255,.10);
                border: 1px solid rgba(255,255,255,.16);
                font-size: .84rem;
                letter-spacing: .04em;
                text-transform: uppercase;
            }

            .hero-shell h1 {
                position: relative;
                z-index: 2;
                margin: 14px 0 10px;
                font-size: 2.7rem;
                line-height: 1;
                letter-spacing: -.03em;
            }

            .hero-shell p {
                position: relative;
                z-index: 2;
                margin: 0;
                max-width: 880px;
                font-size: 1rem;
                line-height: 1.65;
                color: rgba(255,255,255,.86);
            }

            .metrics-ribbon {
                display: grid;
                grid-template-columns: repeat(5, minmax(0, 1fr));
                gap: 14px;
                margin: 18px 0 24px;
            }

            .metric-tile {
                background: rgba(255,255,255,.82);
                border: 1px solid rgba(7, 17, 31, 0.08);
                box-shadow: 0 18px 44px rgba(7, 17, 31, 0.08);
                backdrop-filter: blur(12px);
                border-radius: 22px;
                padding: 16px 18px;
                animation: fadeLift .7s ease both;
            }

            .metric-tile span {
                display: block;
                color: #52606d;
                font-size: .8rem;
                margin-bottom: 6px;
            }

            .metric-tile strong {
                display: block;
                color: #07111f;
                font-size: 1.3rem;
                font-weight: 800;
            }

            .section-card {
                background: rgba(255,255,255,.86);
                border: 1px solid rgba(7, 17, 31, 0.08);
                border-radius: 24px;
                padding: 20px 20px 12px;
                box-shadow: 0 20px 50px rgba(7, 17, 31, 0.08);
                backdrop-filter: blur(12px);
                animation: fadeLift .75s ease both;
            }

            .section-title {
                margin: 0 0 4px;
                color: #07111f;
                font-size: 1.22rem;
                font-weight: 800;
            }

            .section-copy {
                margin: 0 0 14px;
                color: #52606d;
                font-size: .95rem;
                line-height: 1.6;
            }

            .insight-box {
                border-radius: 20px;
                padding: 16px 18px;
                background: linear-gradient(135deg, rgba(15,118,110,.08), rgba(8,145,178,.08));
                border: 1px solid rgba(15,118,110,.14);
                margin-bottom: 14px;
            }

            .insight-box strong {
                display: block;
                color: #0f172a;
                margin-bottom: 5px;
            }

            .insight-box span {
                color: #475569;
                line-height: 1.55;
                font-size: .93rem;
            }

            .live-result {
                position: relative;
                overflow: hidden;
                border-radius: 26px;
                padding: 18px 20px;
                margin-bottom: 16px;
                border: 1px solid rgba(14,165,233,.14);
                background: linear-gradient(135deg, rgba(255,255,255,.92), rgba(224,242,254,.88));
                box-shadow: 0 18px 40px rgba(14,165,233,.12);
                animation: pulseCard 2.2s ease-in-out infinite;
            }

            .live-result.risk-high {
                background: linear-gradient(135deg, rgba(255,255,255,.95), rgba(254,226,226,.92));
                border-color: rgba(239,68,68,.18);
            }

            .live-result.risk-low {
                background: linear-gradient(135deg, rgba(255,255,255,.95), rgba(220,252,231,.90));
                border-color: rgba(34,197,94,.18);
            }

            .live-result::before,
            .live-result::after {
                content: "";
                position: absolute;
                left: -10%;
                width: 120%;
                height: 70px;
                border-radius: 44%;
                opacity: .18;
                pointer-events: none;
            }

            .live-result::before {
                bottom: -24px;
                background: linear-gradient(90deg, #38bdf8, #0ea5e9, #14b8a6);
                animation: waveDriftA 7s linear infinite;
            }

            .live-result::after {
                bottom: -34px;
                background: linear-gradient(90deg, #fde68a, #f59e0b, #fb7185);
                animation: waveDriftB 10s linear infinite;
            }

            .live-result h3 {
                margin: 0 0 6px;
                font-size: 1.35rem;
                color: #0f172a;
            }

            .live-result p {
                margin: 0;
                color: #475569;
            }

            .reason-card {
                border-radius: 18px;
                padding: 14px 16px;
                background: rgba(255,255,255,.86);
                border: 1px solid rgba(15,23,42,.08);
                box-shadow: 0 14px 30px rgba(15,23,42,.06);
                margin-bottom: 10px;
                animation: fadeLift .55s ease both;
            }

            .reason-card strong {
                color: #0f172a;
                display: block;
                margin-bottom: 4px;
            }

            .reason-card span {
                color: #475569;
                font-size: .92rem;
                line-height: 1.5;
            }

            div.stButton > button {
                min-height: 50px;
                border-radius: 15px;
                border: 0;
                font-weight: 800;
                background: linear-gradient(90deg, #0f766e, #0ea5e9);
                color: white;
                box-shadow: 0 16px 30px rgba(14, 165, 233, .22);
                transition: transform .16s ease, box-shadow .16s ease;
            }

            div.stButton > button:hover {
                transform: translateY(-2px);
                box-shadow: 0 22px 36px rgba(14, 165, 233, .30);
            }

            @keyframes fadeLift {
                from { opacity: 0; transform: translateY(14px); }
                to { opacity: 1; transform: translateY(0); }
            }

            @keyframes floatA {
                from { transform: translate3d(0, 0, 0); }
                to { transform: translate3d(16px, 18px, 0); }
            }

            @keyframes floatB {
                from { transform: translate3d(0, 0, 0); }
                to { transform: translate3d(-18px, -10px, 0); }
            }

            @media (max-width: 1100px) {
                .metrics-ribbon {
                    grid-template-columns: repeat(2, minmax(0, 1fr));
                }
            }

            @media (max-width: 700px) {
                .metrics-ribbon {
                    grid-template-columns: 1fr;
                }
                .hero-shell h1 {
                    font-size: 2rem;
                }
            }

            .welcome-overlay {
                position: fixed;
                inset: 0;
                z-index: 999999;
                display: flex;
                align-items: center;
                justify-content: center;
                background:
                    radial-gradient(circle at 50% 20%, rgba(56,189,248,.18), transparent 28%),
                    linear-gradient(135deg, #04111f 0%, #0b3954 48%, #0f766e 100%);
                animation: overlayFade 3.4s ease forwards;
                pointer-events: none;
            }

            .welcome-card {
                text-align: center;
                color: white;
                padding: 30px 34px;
                border-radius: 28px;
                background: rgba(255,255,255,.08);
                border: 1px solid rgba(255,255,255,.16);
                backdrop-filter: blur(10px);
                box-shadow: 0 30px 80px rgba(0,0,0,.20);
            }

            .hotel-graphic {
                position: relative;
                width: 170px;
                height: 150px;
                margin: 0 auto 14px;
                animation: hotelFloat 1.8s ease-in-out infinite alternate;
            }

            .hotel-building {
                position: absolute;
                left: 50%;
                bottom: 14px;
                transform: translateX(-50%);
                width: 110px;
                height: 102px;
                border-radius: 14px 14px 8px 8px;
                background: linear-gradient(180deg, #f8fafc 0%, #dbeafe 100%);
                box-shadow: 0 18px 40px rgba(0,0,0,.18);
            }

            .hotel-roof {
                position: absolute;
                left: 50%;
                top: 8px;
                transform: translateX(-50%);
                width: 130px;
                height: 28px;
                border-radius: 18px 18px 8px 8px;
                background: linear-gradient(90deg, #f59e0b, #fb7185);
            }

            .hotel-sign {
                position: absolute;
                left: 50%;
                top: 18px;
                transform: translateX(-50%);
                padding: 4px 10px;
                border-radius: 999px;
                background: #0f766e;
                color: white;
                font-size: .72rem;
                font-weight: 800;
                letter-spacing: .14em;
                text-transform: uppercase;
            }

            .hotel-windows {
                position: absolute;
                left: 50%;
                top: 44px;
                transform: translateX(-50%);
                width: 72px;
                display: grid;
                grid-template-columns: repeat(3, 1fr);
                gap: 8px;
            }

            .hotel-window {
                width: 18px;
                height: 18px;
                border-radius: 4px;
                background: linear-gradient(180deg, #fde68a, #f59e0b);
                box-shadow: 0 0 16px rgba(245,158,11,.35);
                animation: windowGlow 1.6s ease-in-out infinite alternate;
            }

            .hotel-window:nth-child(2),
            .hotel-window:nth-child(5) {
                animation-delay: .4s;
            }

            .hotel-window:nth-child(3),
            .hotel-window:nth-child(6) {
                animation-delay: .8s;
            }

            .hotel-door {
                position: absolute;
                left: 50%;
                bottom: 0;
                transform: translateX(-50%);
                width: 26px;
                height: 34px;
                border-radius: 8px 8px 0 0;
                background: linear-gradient(180deg, #0f172a, #334155);
            }

            .hotel-base {
                position: absolute;
                left: 50%;
                bottom: 0;
                transform: translateX(-50%);
                width: 150px;
                height: 10px;
                border-radius: 999px;
                background: rgba(255,255,255,.18);
            }

            .welcome-title {
                font-size: 2rem;
                font-weight: 800;
                letter-spacing: -.02em;
                margin-bottom: 8px;
                color: #f8fafc;
            }

            .welcome-copy {
                color: rgba(255,255,255,.84);
                font-size: 1rem;
            }

            .welcome-bar {
                width: 260px;
                height: 10px;
                margin: 16px auto 0;
                border-radius: 999px;
                overflow: hidden;
                background: rgba(255,255,255,.12);
            }

            .welcome-bar::after {
                content: "";
                display: block;
                height: 100%;
                width: 45%;
                border-radius: 999px;
                background: linear-gradient(90deg, #fde68a, #38bdf8, #34d399);
                animation: loadingSweep 2.4s ease-in-out infinite;
            }

            @keyframes hotelFloat {
                from { transform: translateY(0px) scale(1); }
                to { transform: translateY(-8px) scale(1.03); }
            }

            @keyframes loadingSweep {
                0% { transform: translateX(-140%); }
                100% { transform: translateX(320%); }
            }

            @keyframes windowGlow {
                from { opacity: .62; }
                to { opacity: 1; }
            }

            @keyframes overlayFade {
                0%, 70% { opacity: 1; visibility: visible; }
                100% { opacity: 0; visibility: hidden; }
            }

            @keyframes pulseCard {
                0%, 100% { transform: translateY(0px); box-shadow: 0 18px 40px rgba(14,165,233,.12); }
                50% { transform: translateY(-3px); box-shadow: 0 24px 50px rgba(14,165,233,.18); }
            }

            @keyframes waveDriftA {
                from { transform: translateX(-4%) rotate(0deg); }
                to { transform: translateX(4%) rotate(360deg); }
            }

            @keyframes waveDriftB {
                from { transform: translateX(5%) rotate(360deg); }
                to { transform: translateX(-5%) rotate(0deg); }
            }
            </style>
            """,
            unsafe_allow_html=True,
        )

    @staticmethod
    def hero(metadata: Dict[str, Any]) -> None:
        st.markdown(
            f"""
            <div class="hero-shell">
                <div class="hero-topline">Benchmark Dashboard | Prediction Console | SHAP Explainability</div>
                <h1>Hotel Cancellation Intelligence</h1>
                <p>
                    Professional dashboard for comparing all trained models, reviewing saved holdout and
                    5-fold validation metrics, inspecting confusion matrices and SHAP explanations, and
                    running live booking predictions from the saved deployment model.
                </p>
            </div>
            """,
            unsafe_allow_html=True,
        )

        cards = {
            "Best Benchmark": metadata.get("best_model", "N/A"),
            "Cloud Model": metadata.get("deployment_model", metadata.get("best_model", "N/A")),
            "Train / Test": f"{int(metadata.get('train_ratio', 0.7) * 100)}% / {int(metadata.get('test_ratio', 0.3) * 100)}%",
            "Cross-Validation": f"{metadata.get('cross_validation_folds', 'N/A')}-fold",
            "Pipeline Wall Clock": _format_duration(metadata.get("total_pipeline_wall_clock_sec")),
            "Runtime": f"Py {metadata.get('python_version', 'N/A')} / TF {metadata.get('tensorflow_version', 'N/A') or 'off'}",
        }
        html = "".join(
            f'<div class="metric-tile"><span>{label}</span><strong>{value}</strong></div>'
            for label, value in cards.items()
        )
        st.markdown(f'<div class="metrics-ribbon">{html}</div>', unsafe_allow_html=True)

    @staticmethod
    def welcome_overlay() -> None:
        st.markdown(
            """
            <div class="welcome-overlay">
                <div class="welcome-card">
                    <div class="hotel-graphic">
                        <div class="hotel-roof"></div>
                        <div class="hotel-building">
                            <div class="hotel-sign">Hotel</div>
                            <div class="hotel-windows">
                                <div class="hotel-window"></div>
                                <div class="hotel-window"></div>
                                <div class="hotel-window"></div>
                                <div class="hotel-window"></div>
                                <div class="hotel-window"></div>
                                <div class="hotel-window"></div>
                            </div>
                            <div class="hotel-door"></div>
                        </div>
                        <div class="hotel-base"></div>
                    </div>
                    <div class="welcome-title">Welcome To Smart Hotel Cancellation Prediction</div>
                    <div class="welcome-copy">Preparing intelligent risk analytics, explainability, and live prediction tools.</div>
                    <div class="welcome-bar"></div>
                </div>
            </div>
            """,
            unsafe_allow_html=True,
        )


class PredictionApp:
    def __init__(self, artifacts_dir: Path = ARTIFACTS_DIR) -> None:
        self.artifacts_dir = artifacts_dir
        self.processor = HotelDataProcessor()
        self.feature_preset = "high_score"

    def add_engineered_features_compat(self, frame: pd.DataFrame) -> pd.DataFrame:
        """Handle both current and older processor method signatures."""
        try:
            return self.processor.add_engineered_features(
                frame,
                feature_preset=self.feature_preset,
            )
        except TypeError:
            return self.processor.add_engineered_features(frame)

    @staticmethod
    def get_expected_columns(model: Any) -> List[str]:
        try:
            preprocessor = model.named_steps["preprocessor"]
            feature_names = getattr(preprocessor, "feature_names_in_", None)
            if feature_names is None:
                return []
            return list(feature_names)
        except Exception:
            return []

    @staticmethod
    def get_preprocessor_column_groups(model: Any) -> tuple[set[str], set[str]]:
        numeric_columns: set[str] = set()
        categorical_columns: set[str] = set()
        try:
            preprocessor = model.named_steps["preprocessor"]
            for name, _transformer, columns in getattr(preprocessor, "transformers_", []):
                if isinstance(columns, str):
                    continue
                if name == "numeric":
                    numeric_columns.update(columns)
                elif name == "categorical":
                    categorical_columns.update(columns)
        except Exception:
            pass
        return numeric_columns, categorical_columns

    def align_input_to_model(
        self,
        engineered_input: pd.DataFrame,
        raw_input: pd.DataFrame,
        model: Any,
        examples: pd.DataFrame,
    ) -> pd.DataFrame:
        expected_columns = self.get_expected_columns(model)
        if not expected_columns:
            return engineered_input

        numeric_columns, categorical_columns = self.get_preprocessor_column_groups(model)
        aligned = engineered_input.copy()

        # Backfill a few known engineered aliases that older artifact sets still expect.
        if "has_agent" in expected_columns and "has_agent" not in aligned.columns and "agent" in raw_input.columns:
            agent_values = pd.to_numeric(raw_input["agent"], errors="coerce").fillna(0)
            aligned["has_agent"] = (agent_values > 0).astype(int)
        if "has_company" in expected_columns and "has_company" not in aligned.columns and "company" in raw_input.columns:
            company_values = pd.to_numeric(raw_input["company"], errors="coerce").fillna(0)
            aligned["has_company"] = (company_values > 0).astype(int)
        if "country_grouped" in expected_columns and "country_grouped" not in aligned.columns and "country" in raw_input.columns:
            country = raw_input["country"].astype(str).fillna("Unknown")
            aligned["country_grouped"] = np.where(
                country.eq("PRT"),
                "PRT",
                np.where(country.eq("GBR"), "GBR", "Other"),
            )

        for column in expected_columns:
            if column in aligned.columns:
                continue
            if column in raw_input.columns:
                aligned[column] = raw_input[column]
                continue
            if column in examples.columns and not examples.empty:
                sample = examples[column]
                if column in numeric_columns:
                    aligned[column] = pd.to_numeric(sample, errors="coerce").fillna(0).median()
                else:
                    mode = sample.astype(str).mode(dropna=True)
                    aligned[column] = mode.iloc[0] if not mode.empty else "Unknown"
                continue
            aligned[column] = 0 if column in numeric_columns else "Unknown"

        aligned = aligned.reindex(columns=expected_columns)
        for column in aligned.columns:
            if column in numeric_columns:
                aligned[column] = pd.to_numeric(aligned[column], errors="coerce").fillna(0)
            elif column in categorical_columns:
                aligned[column] = aligned[column].astype(str).fillna("Unknown")
        return aligned

    def build_model_input(self, raw_input: pd.DataFrame, model: Any, examples: pd.DataFrame) -> pd.DataFrame:
        engineered = self.add_engineered_features_compat(raw_input.copy())
        return self.align_input_to_model(engineered, raw_input, model, examples)

    @staticmethod
    def booking_profile_items(model_input: pd.DataFrame) -> List[tuple[str, str]]:
        row = model_input.iloc[0]
        mapping = [
            ("lead_time", "Booking Window"),
            ("number_of_total_nights", "Stay Length"),
            ("number_of_children_and_adults", "Traveler Count"),
            ("first_time_visitor", "Guest History"),
        ]
        items: List[tuple[str, str]] = []
        for column, label in mapping:
            if column not in row.index:
                continue
            value = row[column]
            items.append((label, PredictionApp.format_field_value(column, value)))
        return items

    @staticmethod
    def display_label(column_name: str) -> str:
        return FIELD_LABELS.get(column_name, column_name.replace("_", " ").title())

    @staticmethod
    def format_field_value(column_name: str, value: Any) -> str:
        option_map = FIELD_OPTION_LABELS.get(column_name)
        if option_map:
            try:
                numeric_value = int(float(value))
            except (TypeError, ValueError):
                numeric_value = None
            if numeric_value in option_map:
                return option_map[numeric_value]
        if isinstance(value, float):
            return f"{value:.2f}"
        return str(value)

    @staticmethod
    def file_version(path: Path) -> int:
        if not path.exists():
            return 0
        return path.stat().st_mtime_ns

    @st.cache_data(show_spinner=False)
    def load_json(_self, path: Path, default: Any, version: int) -> Any:
        if not path.exists():
            return default
        return json.loads(path.read_text(encoding="utf-8-sig"))

    @st.cache_data(show_spinner=False)
    def load_csv(_self, path: Path, version: int) -> pd.DataFrame:
        if not path.exists():
            return pd.DataFrame()
        return pd.read_csv(path)

    @st.cache_data(show_spinner=False)
    def load_raw_data(_self, path: Path, version: int) -> pd.DataFrame:
        if not path.exists():
            return pd.DataFrame()
        return pd.read_csv(path)

    @st.cache_resource(show_spinner=False)
    def load_models(_self, artifacts_dir: Path, model_names: tuple[str, ...], version: int) -> Dict[str, Any]:
        models: Dict[str, Any] = {}
        models_dir = artifacts_dir / "models"
        if not models_dir.exists():
            return models
        for model_name in model_names:
            slug_name = model_name.lower().replace(" ", "_")
            model_path = models_dir / f"{slug_name}.joblib"
            if model_path.exists():
                models[model_name] = joblib.load(model_path)
        deployment_path = models_dir / "deployment_model.joblib"
        deployment_name = str(model_names[0]) if model_names else "Deployment Model"
        if deployment_path.exists():
            models[deployment_name] = joblib.load(deployment_path)
        return models

    def run(self) -> None:
        st.set_page_config(page_title="Hotel Cancellation Intelligence", layout="wide")
        DashboardStyle.apply()
        if "welcome_seen" not in st.session_state:
            DashboardStyle.welcome_overlay()
            st.session_state["welcome_seen"] = True
            time.sleep(2.8)
            st.rerun()

        self.artifacts_dir = ARTIFACTS_DIR

        metadata_path = self.artifacts_dir / "reports" / "metadata.json"
        confusion_path = self.artifacts_dir / "reports" / "confusion_matrices.json"
        schema_path = self.artifacts_dir / "reports" / "prediction_schema.json"
        examples_path = self.artifacts_dir / "reports" / "prediction_examples.csv"
        holdout_path = self.artifacts_dir / "reports" / "holdout_summary.csv"
        cv_path = self.artifacts_dir / "reports" / "cross_validation_results.csv"
        segments_path = self.artifacts_dir / "reports" / "guest_segments.csv"
        model_path = self.artifacts_dir / "models" / "deployment_model.joblib"

        metadata = self.load_json(metadata_path, {}, self.file_version(metadata_path))
        confusion_data = self.load_json(confusion_path, {}, self.file_version(confusion_path))
        schema = self.load_json(schema_path, {"columns": []}, self.file_version(schema_path))
        examples = self.load_csv(examples_path, self.file_version(examples_path))
        holdout = self.load_csv(holdout_path, self.file_version(holdout_path))
        cv_results = self.load_csv(cv_path, self.file_version(cv_path))
        guest_segments = self.load_csv(segments_path, self.file_version(segments_path))
        metadata_data_path = Path(str(metadata.get("data_path", DATA_PATH)))
        if not metadata_data_path.exists():
            metadata_data_path = DATA_PATH
        raw_data = self.load_raw_data(metadata_data_path, self.file_version(metadata_data_path))
        ordered_model_names: List[str] = []
        deployment_name = str(metadata.get("deployment_model", metadata.get("best_model", "Deployment Model")))
        trained_names = list(metadata.get("full_data_models") or metadata.get("trained_models") or [])
        if deployment_name:
            ordered_model_names.append(deployment_name)
        for name in trained_names:
            if name not in ordered_model_names:
                ordered_model_names.append(name)
        models = self.load_models(
            self.artifacts_dir,
            tuple(ordered_model_names),
            self.file_version(model_path),
        )

        if holdout.empty or not schema.get("columns"):
            st.error("Saved training artifacts are missing. Run terminal training and redeploy the app.")
            return

        schema = self.sanitize_schema(schema)
        examples = self.sanitize_examples(examples)
        holdout = self.normalize_holdout_frame(holdout)
        cv_results = self.normalize_cv_frame(cv_results)
        holdout = self.add_complexity_tiers(holdout)
        self.feature_preset = str(metadata.get("feature_preset", "high_score"))

        DashboardStyle.hero(metadata)
        st.info(
            "Current app mode uses the notebook-aligned reservation benchmark only. The form, evaluation tables, deployed model, and reports all come from the same reservation artifact set."
        )

        overview_tab, models_tab, segment_tab, explain_tab, predict_tab = st.tabs(
            ["Overview", "Model Comparison", "Guest Segmentation", "Explainability", "Prediction"]
        )

        with overview_tab:
            self.render_overview(holdout, cv_results, guest_segments, metadata, raw_data)

        with models_tab:
            self.render_model_comparison(holdout, cv_results, confusion_data)

        with segment_tab:
            self.render_segmentation(guest_segments)

        with explain_tab:
            self.render_explainability()

        with predict_tab:
            self.render_prediction_console(models, schema, examples, metadata)

    @staticmethod
    def normalize_holdout_frame(holdout: pd.DataFrame) -> pd.DataFrame:
        frame = holdout.copy()
        defaults = {
            "complexity_score": np.nan,
            "model_size_mb": np.nan,
            "train_accuracy": np.nan,
            "train_precision": np.nan,
            "train_recall": np.nan,
            "train_f1": np.nan,
            "train_balanced_accuracy": np.nan,
            "train_average_precision": np.nan,
            "train_brier_score": np.nan,
            "train_log_loss": np.nan,
            "train_mcc": np.nan,
            "train_roc_auc": np.nan,
            "accuracy": np.nan,
            "precision": np.nan,
            "recall": np.nan,
            "f1": np.nan,
            "balanced_accuracy": np.nan,
            "average_precision": np.nan,
            "brier_score": np.nan,
            "log_loss": np.nan,
            "mcc": np.nan,
            "roc_auc": np.nan,
            "transformed_feature_count": np.nan,
            "training_time_sec": np.nan,
            "benchmark_training_time_sec": np.nan,
            "full_data_training_time_sec": np.nan,
            "inference_time_sec": np.nan,
            "inference_ms_per_row": np.nan,
        }
        for column, value in defaults.items():
            if column not in frame.columns:
                frame[column] = value
        return frame

    @staticmethod
    def normalize_cv_frame(cv_results: pd.DataFrame) -> pd.DataFrame:
        if cv_results.empty:
            return cv_results
        frame = cv_results.copy()
        for column in ["accuracy", "precision", "recall", "f1", "balanced_accuracy", "roc_auc", "average_precision"]:
            if column not in frame.columns:
                frame[column] = np.nan
        return frame

    @staticmethod
    def add_complexity_tiers(holdout: pd.DataFrame) -> pd.DataFrame:
        frame = holdout.copy()
        required = ["training_time_sec", "inference_ms_per_row", "transformed_feature_count"]
        for column in required:
            values = pd.to_numeric(frame[column], errors="coerce")
            fill_value = float(values.median()) if not values.dropna().empty else 0.0
            frame[column] = values.fillna(fill_value)
        scoring = (
            frame["training_time_sec"].rank(pct=True).fillna(0.5) * 0.5
            + frame["inference_ms_per_row"].rank(pct=True).fillna(0.5) * 0.3
            + frame["transformed_feature_count"].rank(pct=True).fillna(0.5) * 0.2
        )
        frame["complexity_tier"] = pd.cut(
            scoring,
            bins=[-0.01, 0.34, 0.67, 1.01],
            labels=["Low", "Medium", "High"],
        ).astype(str)
        return frame

    @staticmethod
    def sanitize_schema(schema: Dict[str, Any]) -> Dict[str, Any]:
        columns = [column for column in schema.get("columns", []) if column.get("name") not in EXCLUDED_FEATURES]
        return {"columns": columns}

    @staticmethod
    def sanitize_examples(examples: pd.DataFrame) -> pd.DataFrame:
        if examples.empty:
            return examples
        return examples.drop(columns=[col for col in EXCLUDED_FEATURES if col in examples.columns], errors="ignore")

    def render_section_header(self, title: str, copy: str) -> None:
        st.markdown(
            f"""
            <div class="section-card">
                <div class="section-title">{title}</div>
                <div class="section-copy">{copy}</div>
            </div>
            """,
            unsafe_allow_html=True,
        )

    def render_overview(
        self,
        holdout: pd.DataFrame,
        cv_results: pd.DataFrame,
        guest_segments: pd.DataFrame,
        metadata: Dict[str, Any],
        raw_data: pd.DataFrame,
    ) -> None:
        self.render_section_header(
            "Performance Overview",
            "This section summarizes the saved benchmark for the active mode, including holdout metrics, validation consistency, timing behavior, and guest segmentation outputs.",
        )

        top_model = holdout.sort_values("f1", ascending=False).iloc[0]
        secondary = holdout.sort_values("roc_auc", ascending=False).iloc[0]
        insight_left, insight_right = st.columns(2, gap="large")
        with insight_left:
            st.markdown(
                f"""
                <div class="insight-box">
                    <strong>Best holdout performer</strong>
                    <span>{top_model['model']} leads the active {int(metadata.get('test_ratio', 0.2) * 100)}% test split with test accuracy {top_model['accuracy']:.4f}, test F1 {top_model['f1']:.4f}, test ROC-AUC {top_model['roc_auc']:.4f}, and benchmark train accuracy {top_model.get('train_accuracy', np.nan):.4f}.</span>
                </div>
                """,
                unsafe_allow_html=True,
            )
        with insight_right:
            st.markdown(
                f"""
                <div class="insight-box">
                    <strong>Best ranking power</strong>
                    <span>{secondary['model']} shows the strongest class ranking signal with ROC-AUC {secondary['roc_auc']:.4f}. Training and inference times shown below come from the real reservation benchmark run.</span>
                </div>
                """,
                unsafe_allow_html=True,
            )

        if metadata.get("pipeline_wall_clock_note"):
            timing_note = (
                f"{metadata['pipeline_wall_clock_note']} "
                f"Saved pipeline wall clock: {_format_duration(metadata.get('total_pipeline_wall_clock_sec'))}."
            )
        elif metadata.get("total_pipeline_wall_clock_sec") is not None:
            timing_note = (
                f"Total pipeline wall clock for the saved run was {_format_duration(metadata.get('total_pipeline_wall_clock_sec'))}. "
                f"This includes benchmark training, cross-validation, SHAP generation, full-data retraining, and artifact creation."
            )
        else:
            timing_note = (
                "Training time in the tables refers to the saved benchmark run and may not match the total interactive experimentation time."
            )
        st.caption(timing_note)

        left, right = st.columns([1.25, 0.75], gap="large")
        with left:
            st.plotly_chart(self.build_metric_radar(top_model), use_container_width=True)
        with right:
            cv_mean = cv_results[cv_results["fold"].astype(str) == "mean"].sort_values("f1", ascending=False)
            if cv_mean.empty:
                st.info("Cross-validation results are not available for the current artifact set.")
            else:
                st.plotly_chart(self.build_cv_f1_chart(cv_mean), use_container_width=True)

        rnn_reason = metadata.get("skipped_models", {}).get("RNN")
        if rnn_reason:
            st.markdown(
                f"""
                <div class="insight-box">
                    <strong>RNN status</strong>
                    <span>RNN is not included in the current benchmark tables because it was not trainable in this deployment environment. Reason: {rnn_reason}</span>
                </div>
                """,
                unsafe_allow_html=True,
            )

        chart_left, chart_right = st.columns(2, gap="large")
        with chart_left:
            st.plotly_chart(self.build_metric_heatmap(holdout), use_container_width=True)
        with chart_right:
            st.plotly_chart(self.build_timing_combo_chart(holdout), use_container_width=True)

        if not guest_segments.empty:
            seg_left, seg_right = st.columns([1, 1], gap="large")
            with seg_left:
                self.render_image_card(
                    "Guest Segmentation",
                    "K-Means projection of guest groups built from booking behavior features.",
                    self.artifacts_dir / "plots" / "guest_segmentation.png",
                )
            with seg_right:
                st.markdown("### Segment Summary")
                st.dataframe(
                    guest_segments.rename(
                        columns={
                            "segment": "segment_id",
                            "segment_name": "segment_name",
                            "lead_time": "booking_window_avg",
                            "average_price": "nightly_rate_avg",
                            "number_of_total_nights": "stay_band_avg",
                            "number_of_children_and_adults": "traveler_count_avg",
                            "special_requests": "request_count_avg",
                            "cancellation_ratio": "history_cancel_share_avg",
                        }
                    ),
                    use_container_width=True,
                )

        with st.expander("Benchmark Metadata"):
            st.json(metadata)

    def render_model_comparison(
        self,
        holdout: pd.DataFrame,
        cv_results: pd.DataFrame,
        confusion_data: Dict[str, Any],
    ) -> None:
        self.render_section_header(
            "Model Comparison",
            "Compare all trained models side by side using holdout metrics, validation averages, model size, timing, and confusion matrices.",
        )

        metric_options = [
            "accuracy",
            "f1",
            "roc_auc",
            "precision",
            "recall",
            "balanced_accuracy",
            "average_precision",
            "brier_score",
            "log_loss",
            "mcc",
            "train_accuracy",
            "train_f1",
            "train_roc_auc",
            "training_time_sec",
            "inference_ms_per_row",
        ]
        metric = st.selectbox("Comparison metric", metric_options, index=0, key="comparison_metric")

        left, right = st.columns([1.1, 0.9], gap="large")
        with left:
            st.plotly_chart(self.build_metrics_comparison_chart(holdout), use_container_width=True)
        with right:
            st.plotly_chart(self.build_accuracy_vs_time(holdout), use_container_width=True)

        st.plotly_chart(self.build_holdout_bar(holdout, metric), use_container_width=True)

        styled = holdout.copy()
        display_columns = ["model"]
        rename_map = {"model": "model"}
        for source_column, display_column in MODEL_METRIC_COLUMNS:
            if source_column in styled.columns:
                display_columns.append(source_column)
                rename_map[source_column] = display_column
        for column in [
            "benchmark_training_time_sec",
            "full_data_training_time_sec",
            "training_time_sec",
            "inference_ms_per_row",
            "complexity_tier",
        ]:
            if column in styled.columns:
                display_columns.append(column)
        styled = styled[display_columns].rename(columns=rename_map)
        numeric_columns = styled.select_dtypes(include=["number"]).columns
        st.dataframe(
            styled.style.format({column: "{:.4f}" for column in numeric_columns}),
            use_container_width=True,
        )
        st.caption("The `test_*` columns come from the saved 80/20 holdout benchmark split. The `train_*` columns come from the corresponding benchmark training split for the same fitted model. `benchmark_training_time_sec` is the benchmark fit time, `full_data_training_time_sec` is the deployment retrain on the full dataset, and `training_time_sec` is their combined saved run cost. `complexity_tier` is derived from measured training time, inference time, and transformed feature count.")

        cv_mean = cv_results[cv_results["fold"].astype(str) == "mean"].copy()
        if not cv_mean.empty:
            st.markdown("### 5-Fold Validation Means")
            cv_mean = cv_mean[
                [
                    "model",
                    "accuracy",
                    "precision",
                    "recall",
                    "f1",
                    "balanced_accuracy",
                    "roc_auc",
                    "average_precision",
                ]
            ]
            st.dataframe(
                cv_mean.style.format(
                    {column: "{:.4f}" for column in cv_mean.select_dtypes(include=["number"]).columns}
                ),
                use_container_width=True,
            )
            if "RNN" not in cv_mean["model"].tolist() and "RNN" in holdout["model"].tolist():
                st.caption("RNN appears in the holdout benchmark, but the saved 5-fold validation table does not yet include an RNN row from the TensorFlow runtime.")

        model_options = holdout["model"].tolist()
        selected_confusion = st.selectbox("Confusion matrix model", model_options, index=0)
        if selected_confusion in confusion_data:
            st.plotly_chart(
                self.build_confusion_heatmap(selected_confusion, confusion_data[selected_confusion]),
                use_container_width=True,
            )
        else:
            confusion_name = selected_confusion.lower().replace(" ", "_")
            self.render_image_card(
                f"{selected_confusion} Confusion Matrix",
                "Saved confusion matrix from the holdout evaluation.",
                self.artifacts_dir / "plots" / f"{confusion_name}_confusion_matrix.png",
            )

    def render_explainability(self) -> None:
        self.render_section_header(
            "Explainability",
            "Live explainability is generated after each prediction. The visuals below update from the last booking you scored, so they always reflect that booking's exact features and result.",
        )
        self.render_live_prediction_explainability(section_key="explain")

    def render_prediction_console(
        self,
        models: Dict[str, Any],
        schema: Dict[str, Any],
        examples: pd.DataFrame,
        metadata: Dict[str, Any],
    ) -> None:
        self.render_section_header(
            "Prediction Console",
            "Use the deployed reservation model to score a booking manually. The form, saved metrics, and loaded model all come from the same notebook-aligned artifact set.",
        )
        left, right = st.columns([1.2, 0.8], gap="large")

        with left:
            st.markdown("### Manual Booking Form")
            model_name = st.selectbox("Prediction model", list(models.keys()))
            input_frame = self.render_form(schema)
            if st.button("Predict Cancellation Risk", type="primary", use_container_width=True):
                with st.spinner("Building live prediction and SHAP explanation..."):
                    self.render_prediction(models[model_name], input_frame, model_name, examples)

        with right:
            st.markdown("### Deployment Snapshot")
            deployment_name = metadata.get("deployment_model", metadata.get("best_model", "N/A"))
            deployment_note = metadata.get("deployment_model_note", "")
            st.markdown(
                f"""
                <div class="insight-box">
                    <strong>Cloud deployment model</strong>
                    <span>{deployment_name} is the active deployed model for the reservation benchmark app. {deployment_note}</span>
                </div>
                """,
                unsafe_allow_html=True,
            )
            if not examples.empty:
                st.markdown("### Example Rows")
                st.dataframe(examples.head(8), use_container_width=True)

        self.render_live_prediction_explainability(section_key="predict")

    def render_segmentation(self, guest_segments: pd.DataFrame) -> None:
        self.render_section_header(
            "Guest Segmentation",
            "K-means clustering groups guests with similar booking behavior. This section shows the saved segmentation plot and the cluster profile table from the training artifacts.",
        )
        left, right = st.columns([1.1, 0.9], gap="large")
        with left:
            self.render_image_card(
                "K-Means Guest Segmentation",
                "Cluster projection built during terminal training from real booking behavior features.",
                self.artifacts_dir / "plots" / "guest_segmentation.png",
            )
        with right:
            if guest_segments.empty:
                st.info("Guest segmentation artifacts are not available yet.")
            else:
                st.plotly_chart(self.build_segmentation_profile_chart(guest_segments), use_container_width=True)
                st.dataframe(
                    guest_segments.rename(
                        columns={
                            "segment": "segment_id",
                            "segment_name": "segment_name",
                            "lead_time": "booking_window_avg",
                            "average_price": "nightly_rate_avg",
                            "number_of_total_nights": "stay_band_avg",
                            "number_of_children_and_adults": "traveler_count_avg",
                            "special_requests": "request_count_avg",
                            "cancellation_ratio": "history_cancel_share_avg",
                        }
                    ),
                    use_container_width=True,
                )

    def render_form(self, schema: Dict[str, Any]) -> pd.DataFrame:
        values: Dict[str, Any] = {}
        columns = st.columns(3)
        for index, column in enumerate(schema.get("columns", [])):
            holder = columns[index % 3]
            field_name = column["name"]
            field_label = self.display_label(field_name)
            option_map = FIELD_OPTION_LABELS.get(field_name, {})
            if column["type"] == "categorical":
                options: List[str] = column["options"]
                default = column["default"]
                default_index = options.index(default) if default in options else 0
                values[field_name] = holder.selectbox(
                    field_label,
                    options=options,
                    index=default_index,
                    key=f"field_{field_name}",
                )
            elif option_map:
                numeric_options = sorted(option_map)
                default_value = int(round(float(column["default"])))
                default_index = numeric_options.index(default_value) if default_value in numeric_options else 0
                selected_option = holder.selectbox(
                    field_label,
                    options=numeric_options,
                    index=default_index,
                    format_func=lambda option: option_map.get(option, str(option)),
                    key=f"field_{field_name}",
                )
                values[field_name] = selected_option
            else:
                values[field_name] = holder.number_input(
                    field_label,
                    min_value=float(column["min"]),
                    max_value=float(column["max"]),
                    value=float(column["default"]),
                    step=float(column["step"]),
                    key=f"field_{field_name}",
                )
        return pd.DataFrame([values])

    def render_prediction(
        self,
        model: Any,
        raw_input: pd.DataFrame,
        model_name: str,
        examples: pd.DataFrame,
    ) -> None:
        engineered_preview = self.add_engineered_features_compat(raw_input.copy())
        model_input = self.align_input_to_model(engineered_preview.copy(), raw_input, model, examples)

        prediction = int(model.predict(model_input)[0])
        probabilities = _positive_probabilities(model, model_input)
        cancel_probability = float(probabilities[0]) if probabilities is not None else None

        st.divider()
        status_col, metric_col = st.columns([1, 1], gap="large")
        with status_col:
            if prediction == 1:
                st.error(f"{model_name}: this booking is predicted to cancel.")
            else:
                st.success(f"{model_name}: this booking is predicted to stay active.")

        if cancel_probability is not None:
            stay_probability = 1 - cancel_probability
            with metric_col:
                st.metric("Cancellation probability", f"{cancel_probability * 100:.2f}%")
                st.metric("Stay probability", f"{stay_probability * 100:.2f}%")
            risk_text = (
                "Risk is elevated, so this reservation deserves proactive retention handling."
                if cancel_probability >= 0.5
                else "Risk is lower, so this reservation currently looks stable."
            )
            st.info(risk_text)

        profile_items = self.booking_profile_items(engineered_preview)
        if profile_items:
            st.markdown("### Booking Profile")
            profile_columns = st.columns(len(profile_items))
            for column, (label, value) in zip(profile_columns, profile_items):
                column.metric(label, value)

        # K-Means Segmentation Matching
        segment_path = self.artifacts_dir / "reports" / "guest_segments.csv"
        if segment_path.exists():
            try:
                segments_df = pd.read_csv(segment_path)
                features_to_match = [
                    column
                    for column in segments_df.columns
                    if column not in {"segment", "segment_name"} and pd.api.types.is_numeric_dtype(segments_df[column])
                ]

                live_values = {
                    column: float(pd.to_numeric(model_input[column], errors="coerce").iloc[0])
                    if column in model_input.columns
                    else 0.0
                    for column in features_to_match
                }

                min_dist = float("inf")
                best_segment = -1
                best_segment_name = ""
                for _, row in segments_df.iterrows():
                    dist = sum(
                        (
                            (float(pd.to_numeric(pd.Series([row[column]]), errors="coerce").iloc[0]) - live_values[column])
                            / max(1.0, abs(float(pd.to_numeric(pd.Series([row[column]]), errors="coerce").iloc[0])))
                        )
                        ** 2
                        for column in features_to_match
                    )
                    if dist < min_dist:
                        min_dist = dist
                        best_segment = int(row["segment"])
                        best_segment_name = str(row.get("segment_name", f"Segment {best_segment}"))

                if best_segment >= 0:
                    st.markdown(
                        f"**Guest Intelligence:** Based on k-means clustering, this booking aligns most closely with **{best_segment_name}**."
                    )
            except Exception:
                pass

        if examples.empty:
            return

        try:
            analyzer = SHAPAnalyzer()
            background_raw = examples.head(min(80, len(examples))).copy()
            background = self.build_model_input(background_raw, model, examples)
            shap_values = analyzer.explain(model, background, model_input, max_background=80)
            feature_names = list(shap_values.feature_names)
            shap_row = np.asarray(shap_values.values[0], dtype=float)
            data_row = np.asarray(shap_values.data[0])
            explanation_frame = pd.DataFrame(
                {
                    "feature": feature_names,
                    "feature_value": data_row,
                    "shap_value": shap_row,
                }
            )
            explanation_frame["feature"] = explanation_frame["feature"].map(self.display_label)
            increasing = explanation_frame[explanation_frame["shap_value"] > 0].nlargest(5, "shap_value")
            decreasing = explanation_frame[explanation_frame["shap_value"] < 0].nsmallest(5, "shap_value")
            st.session_state["latest_prediction"] = {
                "model_name": model_name,
                "prediction": prediction,
                "cancel_probability": cancel_probability,
                "stay_probability": None if cancel_probability is None else 1 - cancel_probability,
                "increasing": increasing.to_dict(orient="records"),
                "decreasing": decreasing.to_dict(orient="records"),
            }
        except Exception as exc:
            st.warning(f"Prediction SHAP explanation is currently unavailable: {exc}")

    def render_image_card(self, title: str, copy: str, path: Path) -> None:
        st.markdown(f"### {title}")
        st.caption(copy)
        if path.exists():
            st.image(str(path), use_container_width=True)
        else:
            st.warning(f"Missing artifact: {path.name}")

    def render_live_prediction_explainability(self, section_key: str) -> None:
        latest = st.session_state.get("latest_prediction")
        if not latest:
            st.markdown(
                """
                <div class="insight-box">
                    <strong>Live SHAP will appear here</strong>
                    <span>Run a prediction to generate interactive feature explanations showing exactly which features increased cancellation risk and which ones reduced it for that booking.</span>
                </div>
                """,
                unsafe_allow_html=True,
            )
            return

        probability = latest.get("cancel_probability")
        prediction = latest.get("prediction")
        risk_class = "risk-high" if prediction == 1 else "risk-low"
        headline = "Booking Likely To Cancel" if prediction == 1 else "Booking Likely To Stay"
        subcopy = (
            f"Live explanation for {latest.get('model_name', 'model')}. Cancellation probability: {probability * 100:.2f}%."
            if probability is not None
            else f"Live explanation for {latest.get('model_name', 'model')}."
        )
        st.markdown(
            f"""
            <div class="live-result {risk_class}">
                <h3>{headline}</h3>
                <p>{subcopy}</p>
            </div>
            """,
            unsafe_allow_html=True,
        )

        gauge_left, gauge_right = st.columns([0.9, 1.1], gap="large")
        with gauge_left:
            if probability is not None:
                st.plotly_chart(
                    self.build_probability_gauge(probability),
                    use_container_width=True,
                    key=f"probability_gauge_{section_key}",
                )
        with gauge_right:
            inc = pd.DataFrame(latest.get("increasing", []))
            dec = pd.DataFrame(latest.get("decreasing", []))
            st.plotly_chart(
                self.build_local_shap_waterfall(inc, dec),
                use_container_width=True,
                key=f"local_shap_waterfall_{section_key}",
            )

        col1, col2 = st.columns(2, gap="large")
        with col1:
            st.plotly_chart(
                self.build_local_shap_chart(pd.DataFrame(latest.get("increasing", [])), "Features That Increased Cancellation Risk"),
                use_container_width=True,
                key=f"increase_shap_chart_{section_key}",
            )
            for item in latest.get("increasing", []):
                st.markdown(
                    f"""
                    <div class="reason-card">
                        <strong>{item['feature']}</strong>
                        <span>This feature increased cancellation risk for this booking because the current value was <code>{item['feature_value']}</code> and it pushed the model upward by {float(item['shap_value']):.4f}.</span>
                    </div>
                    """,
                    unsafe_allow_html=True,
                )
        with col2:
            st.plotly_chart(
                self.build_local_shap_chart(pd.DataFrame(latest.get("decreasing", [])), "Features That Decreased Cancellation Risk"),
                use_container_width=True,
                key=f"decrease_shap_chart_{section_key}",
            )
            for item in latest.get("decreasing", []):
                st.markdown(
                    f"""
                    <div class="reason-card">
                        <strong>{item['feature']}</strong>
                        <span>This feature decreased cancellation risk for this booking because the current value was <code>{item['feature_value']}</code> and it pulled the model downward by {abs(float(item['shap_value'])):.4f}.</span>
                    </div>
                    """,
                    unsafe_allow_html=True,
                )

    def build_holdout_bar(self, holdout: pd.DataFrame, metric: str) -> go.Figure:
        chart = holdout.sort_values(metric, ascending=False).copy()
        chart[metric] = pd.to_numeric(chart[metric], errors="coerce").fillna(0.0)
        fig = px.bar(
            chart,
            x="model",
            y=metric,
            color=metric,
            color_continuous_scale=["#c4f1f9", "#4cc9f0", "#0f766e"],
            text_auto=".3f",
        )
        fig.update_layout(
            height=460,
            xaxis_title="Model",
            yaxis_title=metric.replace("_", " ").title(),
            margin=dict(l=10, r=10, t=20, b=20),
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(248,250,252,0.7)",
        )
        fig.update_traces(marker_line_color="rgba(7,17,31,0.15)", marker_line_width=1.1)
        return fig

    def build_metrics_comparison_chart(self, holdout: pd.DataFrame) -> go.Figure:
        metrics = ["accuracy", "precision", "recall", "f1", "roc_auc"]
        chart = holdout[["model", *metrics]].copy()
        for metric in metrics:
            chart[metric] = pd.to_numeric(chart[metric], errors="coerce").fillna(0.0)
        long_frame = chart.melt(
            id_vars="model",
            value_vars=metrics,
            var_name="metric",
            value_name="score",
        )
        fig = px.bar(
            long_frame,
            x="model",
            y="score",
            color="metric",
            barmode="group",
            color_discrete_sequence=["#0f766e", "#0ea5e9", "#22c55e", "#f59e0b", "#1d4ed8"],
        )
        fig.update_layout(
            height=500,
            yaxis_title="Score",
            xaxis_title="Model",
            margin=dict(l=10, r=10, t=20, b=20),
            legend_title="Metric",
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(248,250,252,0.7)",
        )
        return fig

    def build_accuracy_vs_time(self, holdout: pd.DataFrame) -> go.Figure:
        chart = holdout.copy()
        chart["training_time_sec"] = pd.to_numeric(chart["training_time_sec"], errors="coerce").fillna(0.0)
        chart["accuracy"] = pd.to_numeric(chart["accuracy"], errors="coerce").fillna(0.0)
        chart["f1"] = pd.to_numeric(chart["f1"], errors="coerce").fillna(0.0)
        chart["roc_auc"] = pd.to_numeric(chart["roc_auc"], errors="coerce").fillna(0.0)
        chart["inference_ms_per_row"] = pd.to_numeric(chart["inference_ms_per_row"], errors="coerce").fillna(0.0)
        fig = px.scatter(
            chart,
            x="training_time_sec",
            y="accuracy",
            color="model",
            symbol="complexity_tier" if "complexity_tier" in chart.columns else None,
            hover_data=["f1", "roc_auc", "inference_ms_per_row"],
        )
        fig.update_layout(
            height=460,
            xaxis_title="Training time (seconds)",
            yaxis_title="Holdout accuracy",
            margin=dict(l=10, r=10, t=20, b=20),
            showlegend=False,
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(248,250,252,0.7)",
        )
        fig.update_traces(marker=dict(line=dict(color="white", width=1.2), opacity=0.9))
        return fig

    def build_cv_f1_chart(self, cv_mean: pd.DataFrame) -> go.Figure:
        fig = px.bar(
            cv_mean,
            x="model",
            y="f1",
            color="f1",
            color_continuous_scale=["#dff7f3", "#34d399", "#0f766e"],
            text_auto=".3f",
        )
        fig.update_layout(
            height=420,
            xaxis_title="Model",
            yaxis_title="Mean 5-fold F1",
            margin=dict(l=10, r=10, t=20, b=20),
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(248,250,252,0.7)",
        )
        fig.update_traces(marker_line_color="rgba(7,17,31,0.15)", marker_line_width=1.1)
        return fig

    def build_metric_radar(self, top_model: pd.Series) -> go.Figure:
        metrics = ["accuracy", "precision", "recall", "f1", "roc_auc", "balanced_accuracy"]
        fig = go.Figure()
        fig.add_trace(
            go.Scatterpolar(
                r=[float(top_model[metric]) for metric in metrics],
                theta=[metric.replace("_", " ").title() for metric in metrics],
                fill="toself",
                name=str(top_model["model"]),
                line=dict(color="#0f766e", width=3),
                fillcolor="rgba(15, 118, 110, 0.28)",
            )
        )
        fig.update_layout(
            polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
            height=420,
            margin=dict(l=10, r=10, t=20, b=20),
            showlegend=False,
            paper_bgcolor="rgba(0,0,0,0)",
        )
        return fig

    def build_metric_heatmap(self, holdout: pd.DataFrame) -> go.Figure:
        metrics = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]
        frame = holdout.set_index("model")[metrics].apply(pd.to_numeric, errors="coerce").fillna(0.0)
        fig = px.imshow(
            frame,
            text_auto=".3f",
            aspect="auto",
            color_continuous_scale=["#f8fafc", "#99f6e4", "#0f766e"],
        )
        fig.update_layout(
            title="Metric Comparison Heatmap",
            height=480,
            margin=dict(l=10, r=10, t=40, b=20),
            paper_bgcolor="rgba(0,0,0,0)",
        )
        return fig

    def build_timing_combo_chart(self, holdout: pd.DataFrame) -> go.Figure:
        chart = holdout.sort_values("f1", ascending=False).copy()
        chart["training_time_sec"] = pd.to_numeric(chart["training_time_sec"], errors="coerce").fillna(0.0)
        chart["inference_ms_per_row"] = pd.to_numeric(chart["inference_ms_per_row"], errors="coerce").fillna(0.0)
        fig = make_subplots(specs=[[{"secondary_y": True}]])
        fig.add_trace(
            go.Bar(
                x=chart["model"],
                y=chart["training_time_sec"],
                name="Training time (sec)",
                marker_color="#0ea5e9",
            ),
            secondary_y=False,
        )
        fig.add_trace(
            go.Scatter(
                x=chart["model"],
                y=chart["inference_ms_per_row"],
                mode="lines+markers+text",
                name="Inference ms/row",
                marker=dict(color="#f59e0b", size=10),
                line=dict(color="#f59e0b", width=3),
                text=[f"{value:.3f}" for value in chart["inference_ms_per_row"]],
                textposition="top center",
            ),
            secondary_y=True,
        )
        fig.update_layout(
            title="Measured Training vs Inference Cost",
            height=480,
            margin=dict(l=10, r=10, t=40, b=20),
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(248,250,252,0.7)",
        )
        fig.update_yaxes(title_text="Training time (sec)", secondary_y=False)
        fig.update_yaxes(title_text="Inference ms/row", secondary_y=True)
        return fig

    def build_confusion_heatmap(self, model_name: str, payload: Dict[str, Any]) -> go.Figure:
        matrix = np.asarray(payload["matrix"])
        fig = px.imshow(
            matrix,
            text_auto=True,
            x=payload.get("predicted", ["Predicted 0", "Predicted 1"]),
            y=payload.get("labels", ["Actual 0", "Actual 1"]),
            color_continuous_scale=["#eff6ff", "#38bdf8", "#0f766e"],
            aspect="auto",
        )
        fig.update_layout(
            title=f"{model_name} Confusion Matrix",
            height=430,
            margin=dict(l=10, r=10, t=40, b=20),
            paper_bgcolor="rgba(0,0,0,0)",
        )
        return fig

    def build_global_shap_importance_chart(self, metadata: Dict[str, Any]) -> go.Figure:
        frame = pd.DataFrame(metadata.get("shap_explanations", []))
        if frame.empty:
            frame = pd.DataFrame(
                {"feature": ["No saved SHAP data"], "mean_abs_shap": [0.0]}
            )
        fig = px.bar(
            frame.sort_values("mean_abs_shap", ascending=True),
            x="mean_abs_shap",
            y="feature",
            orientation="h",
            color="mean_abs_shap",
            color_continuous_scale=["#dbeafe", "#38bdf8", "#0f766e"],
            text_auto=".3f",
        )
        fig.update_layout(
            height=420,
            xaxis_title="Mean |SHAP|",
            yaxis_title="Feature",
            margin=dict(l=10, r=10, t=20, b=20),
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(248,250,252,0.7)",
            showlegend=False,
        )
        return fig

    def build_local_shap_chart(self, frame: pd.DataFrame, title: str) -> go.Figure:
        if frame.empty:
            frame = pd.DataFrame({"feature": ["No features"], "shap_value": [0.0]})
        plot_frame = frame.sort_values("shap_value")
        fig = px.bar(
            plot_frame,
            x="shap_value",
            y="feature",
            orientation="h",
            color="shap_value",
            color_continuous_scale=["#22c55e", "#e2e8f0", "#ef4444"],
            text_auto=".3f",
        )
        fig.update_layout(
            title=title,
            height=380,
            xaxis_title="SHAP contribution",
            yaxis_title="Feature",
            margin=dict(l=10, r=10, t=40, b=20),
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(248,250,252,0.7)",
            showlegend=False,
        )
        return fig

    def build_probability_gauge(self, probability: float) -> go.Figure:
        fig = go.Figure(
            go.Indicator(
                mode="gauge+number",
                value=probability * 100,
                number={"suffix": "%", "font": {"size": 36, "color": "#0f172a"}},
                gauge={
                    "axis": {"range": [0, 100]},
                    "bar": {"color": "#0ea5e9"},
                    "steps": [
                        {"range": [0, 35], "color": "#dcfce7"},
                        {"range": [35, 65], "color": "#fef3c7"},
                        {"range": [65, 100], "color": "#fee2e2"},
                    ],
                    "threshold": {"line": {"color": "#ef4444", "width": 4}, "thickness": 0.8, "value": probability * 100},
                },
                title={"text": "Cancellation Probability"},
            )
        )
        fig.update_layout(height=320, margin=dict(l=10, r=10, t=50, b=10), paper_bgcolor="rgba(0,0,0,0)")
        return fig

    def build_segmentation_profile_chart(self, guest_segments: pd.DataFrame) -> go.Figure:
        id_columns = ["segment"]
        color_column = "segment"
        if "segment_name" in guest_segments.columns:
            id_columns.append("segment_name")
            color_column = "segment_name"
        value_columns = [
            column
            for column in guest_segments.columns
            if column not in {"segment", "segment_name"} and pd.api.types.is_numeric_dtype(guest_segments[column])
        ]
        frame = guest_segments.melt(id_vars=id_columns, value_vars=value_columns, var_name="feature", value_name="value")
        frame["feature"] = frame["feature"].map(self.display_label)
        fig = px.bar(
            frame,
            x="feature",
            y="value",
            color=color_column,
            barmode="group",
            color_discrete_sequence=["#0f766e", "#0ea5e9", "#f59e0b", "#ef4444"],
        )
        fig.update_layout(
            title="Cluster Profiles",
            height=420,
            margin=dict(l=10, r=10, t=40, b=20),
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(248,250,252,0.7)",
            xaxis_title="Feature",
            yaxis_title="Average value",
        )
        return fig

    def build_local_shap_waterfall(self, increasing: pd.DataFrame, decreasing: pd.DataFrame) -> go.Figure:
        inc = increasing.copy() if not increasing.empty else pd.DataFrame(columns=["feature", "shap_value"])
        dec = decreasing.copy() if not decreasing.empty else pd.DataFrame(columns=["feature", "shap_value"])
        frame = pd.concat([dec, inc], ignore_index=True)
        if frame.empty:
            frame = pd.DataFrame({"feature": ["No features"], "shap_value": [0.0]})
        frame["label"] = frame["feature"].astype(str)
        fig = go.Figure(
            go.Waterfall(
                orientation="v",
                measure=["relative"] * len(frame),
                x=frame["label"],
                y=frame["shap_value"],
                connector={"line": {"color": "rgba(15,23,42,0.18)"}},
                increasing={"marker": {"color": "#ef4444"}},
                decreasing={"marker": {"color": "#22c55e"}},
            )
        )
        fig.update_layout(
            title="Live SHAP Contribution Flow",
            height=380,
            margin=dict(l=10, r=10, t=40, b=20),
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(248,250,252,0.7)",
            xaxis_title="Feature",
            yaxis_title="Contribution to risk",
        )
        return fig


if __name__ == "__main__":
    PredictionApp().run()


## `hotel_cancellation_oop.py`


In [ ]:
from hotel_app.ml import (
    MONTH_ORDER,
    ANNModel,
    BaseHotelModel,
    DecisionTreeModel,
    EvaluationMetrics,
    HotelDataProcessor,
    KMeansSegmenter,
    KNNModel,
    KerasTabularClassifier,
    LogisticRegressionModel,
    MODEL_REGISTRY,
    ModelTester,
    ModelTrainer,
    NotebookEDAAnalyzer,
    RNNModel,
    RandomForestModel,
    SHAPAnalyzer,
    SVMModel,
    TerminalTrainingRunner,
    TrainingArtifacts,
    ValidationRunner,
    XGBoostModel,
    _count_model_complexity,
    _one_hot_encoder,
    _positive_probabilities,
    _safe_float,
    _slugify,
)

__all__ = [
    "MONTH_ORDER",
    "ANNModel",
    "BaseHotelModel",
    "DecisionTreeModel",
    "EvaluationMetrics",
    "HotelDataProcessor",
    "KMeansSegmenter",
    "KNNModel",
    "KerasTabularClassifier",
    "LogisticRegressionModel",
    "MODEL_REGISTRY",
    "ModelTester",
    "ModelTrainer",
    "NotebookEDAAnalyzer",
    "RNNModel",
    "RandomForestModel",
    "SHAPAnalyzer",
    "SVMModel",
    "TerminalTrainingRunner",
    "TrainingArtifacts",
    "ValidationRunner",
    "XGBoostModel",
    "_count_model_complexity",
    "_one_hot_encoder",
    "_positive_probabilities",
    "_safe_float",
    "_slugify",
]


## `hotel_app/__init__.py`


In [ ]:
from .reporting import BenchmarkPdfBuilder
from .services import MetricsService, TestingService, TrainingService, ValidationService
from .ui import CSSTemplates, HTMLTemplates

__all__ = [
    "BenchmarkPdfBuilder",
    "MetricsService",
    "TestingService",
    "TrainingService",
    "ValidationService",
    "CSSTemplates",
    "HTMLTemplates",
]


## `hotel_app/ui.py`


In [ ]:
from __future__ import annotations


class HTMLTemplates:
    @staticmethod
    def result_card(title: str, copy: str, css_class: str) -> str:
        return f"""
        <div class="live-result {css_class}">
            <h3>{title}</h3>
            <p>{copy}</p>
        </div>
        """


class CSSTemplates:
    @staticmethod
    def wave_card_hint() -> str:
        return ".live-result { overflow: hidden; position: relative; }"


## `hotel_app/services.py`


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

from hotel_cancellation_oop import EvaluationMetrics, ModelTester, ModelTrainer


@dataclass
class TrainingService:
    trainer: ModelTrainer

    def prepare_data(self, *args, **kwargs):
        return self.trainer.prepare_data(*args, **kwargs)

    def split_data(self, *args, **kwargs):
        return self.trainer.split_data(*args, **kwargs)

    def train_model(self, *args, **kwargs):
        return self.trainer.train_model(*args, **kwargs)

    def train_many(self, *args, **kwargs):
        return self.trainer.train_many(*args, **kwargs)


@dataclass
class TestingService:
    tester: ModelTester

    def test_model(self, *args, **kwargs):
        return self.tester.test_model(*args, **kwargs)

    def test_many(self, *args, **kwargs):
        return self.tester.test_many(*args, **kwargs)


@dataclass
class ValidationService:
    trainer: ModelTrainer

    def cross_validate(self, *args, **kwargs):
        return self.trainer.k_fold_cross_validate(*args, **kwargs)


class MetricsService:
    @staticmethod
    def evaluate(*args, **kwargs):
        return EvaluationMetrics.evaluate(*args, **kwargs)

    @staticmethod
    def report(*args, **kwargs):
        return EvaluationMetrics.report(*args, **kwargs)


## `hotel_app/reporting.py`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from textwrap import fill

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import pandas as pd


MODEL_DESCRIPTIONS = {
    "ANN": "Feed-forward neural network used as a dense deep-learning classifier.",
    "LSTM": "Long short-term memory network trained on reshaped tabular sequences with TensorFlow.",
    "KNN": "Distance-based classifier that predicts from nearby training examples.",
    "Decision Tree": "Rule-based tree learner that splits bookings into cancellation patterns.",
    "Logistic Regression": "Linear probabilistic baseline using the logistic sigmoid for binary classification.",
    "Random Forest": "Bagged ensemble of decision trees with randomized feature selection.",
    "SVM": "RBF-kernel support vector machine trained on a bounded stratified sample for practicality.",
    "XGBoost": "Optimized gradient boosting tree method with regularized boosting.",
    "RNN": "Recurrent neural network trained on reshaped tabular sequences with TensorFlow.",
}


def _format_duration(seconds: object) -> str:
    try:
        value = float(seconds)
    except (TypeError, ValueError):
        return "N/A"
    if value < 60:
        return f"{value:.1f}s"
    minutes, remainder = divmod(int(round(value)), 60)
    if minutes < 60:
        return f"{minutes}m {remainder}s"
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h {minutes}m"


class BenchmarkPdfBuilder:
    def __init__(self, artifacts_dir: str | Path = "artifacts") -> None:
        self.artifacts_dir = Path(artifacts_dir)
        self.reports_dir = self.artifacts_dir / "reports"
        self.output_path = self.reports_dir / "model_evaluation_report.pdf"

    def build(self) -> Path:
        holdout = pd.read_csv(self.reports_dir / "holdout_summary.csv")
        cv_results = self._safe_read_csv(self.reports_dir / "cross_validation_results.csv")
        metadata = json.loads((self.reports_dir / "metadata.json").read_text(encoding="utf-8-sig"))

        with PdfPages(self.output_path) as pdf:
            pdf.savefig(self._cover_page(metadata))
            pdf.savefig(self._holdout_page(holdout))
            pdf.savefig(self._cv_page(cv_results))
            pdf.savefig(self._narrative_page(holdout))

        plt.close("all")
        return self.output_path

    @staticmethod
    def _safe_read_csv(path: Path) -> pd.DataFrame:
        try:
            return pd.read_csv(path)
        except (pd.errors.EmptyDataError, FileNotFoundError):
            return pd.DataFrame()

    def _cover_page(self, metadata: dict) -> plt.Figure:
        fig, ax = plt.subplots(figsize=(8.27, 11.69))
        ax.axis("off")
        lines = [
            "Hotel Cancellation Prediction",
            "",
            f"Best benchmark model: {metadata.get('best_model', 'N/A')}",
            f"Deployment artifact: {metadata.get('deployment_model', metadata.get('best_model', 'N/A'))}",
            f"Train / test split: {int(metadata.get('train_ratio', 0.7) * 100)}% / {int(metadata.get('test_ratio', 0.3) * 100)}%",
            f"Cross-validation: {metadata.get('cross_validation_folds', 'N/A')}-fold",
            f"Runtime: Python {metadata.get('python_version', 'N/A')} / TensorFlow {metadata.get('tensorflow_version', 'off')}",
            f"Pipeline wall clock: {_format_duration(metadata.get('total_pipeline_wall_clock_sec'))}",
            "",
            metadata.get(
                "pipeline_wall_clock_note",
                "This report summarizes how each trained model behaved on the dataset using the saved benchmark artifacts.",
            ),
        ]
        ax.text(0.08, 0.92, "\n".join(lines), va="top", ha="left", fontsize=14)
        return fig

    def _holdout_page(self, holdout: pd.DataFrame) -> plt.Figure:
        fig, ax = plt.subplots(figsize=(11.69, 8.27))
        ax.axis("off")
        display = holdout.copy()
        if "complexity_tier" not in display.columns:
            display["complexity_tier"] = "Derived in dashboard"
        cols = [
            c
            for c in [
                "model",
                "train_accuracy",
                "accuracy",
                "train_precision",
                "precision",
                "train_recall",
                "recall",
                "train_f1",
                "f1",
                "train_balanced_accuracy",
                "balanced_accuracy",
                "train_mcc",
                "mcc",
                "train_roc_auc",
                "roc_auc",
                "train_average_precision",
                "average_precision",
                "train_brier_score",
                "brier_score",
                "train_log_loss",
                "log_loss",
                "benchmark_training_time_sec",
                "full_data_training_time_sec",
                "training_time_sec",
                "inference_ms_per_row",
                "complexity_tier",
            ]
            if c in display.columns
        ]
        table = ax.table(
            cellText=display[cols].round(4).astype(str).values,
            colLabels=cols,
            loc="center",
            cellLoc="center",
        )
        table.auto_set_font_size(False)
        table.set_fontsize(7)
        table.scale(1, 1.4)
        ax.set_title("Holdout Evaluation Summary", fontsize=14, pad=18)
        return fig

    def _cv_page(self, cv_results: pd.DataFrame) -> plt.Figure:
        fig, ax = plt.subplots(figsize=(11.69, 8.27))
        ax.axis("off")
        if cv_results.empty or "fold" not in cv_results.columns:
            ax.text(
                0.08,
                0.88,
                "Cross-validation was skipped for this saved run, so no CV rows are available.",
                fontsize=13,
                va="top",
            )
            ax.set_title("Cross-Validation Mean Scores", fontsize=14, pad=18)
            return fig
        means = cv_results[cv_results["fold"].astype(str) == "mean"].copy()
        cols = [c for c in ["model", "accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"] if c in means.columns]
        table = ax.table(
            cellText=means[cols].round(4).astype(str).values,
            colLabels=cols,
            loc="center",
            cellLoc="center",
        )
        table.auto_set_font_size(False)
        table.set_fontsize(8)
        table.scale(1, 1.5)
        ax.set_title("Cross-Validation Mean Scores", fontsize=14, pad=18)
        return fig

    def _narrative_page(self, holdout: pd.DataFrame) -> plt.Figure:
        fig, ax = plt.subplots(figsize=(8.27, 11.69))
        ax.axis("off")
        top = 0.97
        ax.text(0.06, top, "Model-by-model notes", fontsize=16, va="top")
        y = top - 0.05
        for row in holdout.itertuples(index=False):
            description = MODEL_DESCRIPTIONS.get(row.model, "Model description not available.")
            line = (
                f"{row.model}: {description} Benchmark train accuracy {getattr(row, 'train_accuracy', float('nan')):.4f}, "
                f"holdout accuracy {row.accuracy:.4f}, precision {row.precision:.4f}, recall {row.recall:.4f}, "
                f"F1 {row.f1:.4f}, balanced accuracy {getattr(row, 'balanced_accuracy', float('nan')):.4f}, MCC {getattr(row, 'mcc', float('nan')):.4f}, "
                f"ROC-AUC {row.roc_auc:.4f}, average precision {getattr(row, 'average_precision', float('nan')):.4f}, "
                f"benchmark fit {getattr(row, 'benchmark_training_time_sec', float('nan')):.2f}s, "
                f"full-data retrain {getattr(row, 'full_data_training_time_sec', float('nan')):.2f}s, total saved run cost {row.training_time_sec:.2f}s, "
                f"inference {row.inference_ms_per_row:.4f} ms/row."
            )
            wrapped = fill(line, width=90)
            ax.text(0.06, y, wrapped, fontsize=9, va="top")
            y -= 0.08 + 0.02 * wrapped.count("\n")
            if y < 0.08:
                break
        return fig


## `hotel_app/ml/__init__.py`


In [ ]:
from .data import (
    MONTH_ORDER,
    HotelDataProcessor,
    NotebookEDAAnalyzer,
    _count_model_complexity,
    _one_hot_encoder,
    _positive_probabilities,
    _safe_float,
    _slugify,
)
from .deep import KerasTabularClassifier
from .explainability import SHAPAnalyzer
from .metrics import EvaluationMetrics
from .models import (
    ANNModel,
    BaseHotelModel,
    DecisionTreeModel,
    KNNModel,
    LSTMModel,
    LogisticRegressionModel,
    MODEL_REGISTRY,
    RandomForestModel,
    RNNModel,
    SVMModel,
    XGBoostModel,
)
from .testing import ModelTester
from .training import KMeansSegmenter, ModelTrainer, TerminalTrainingRunner, TrainingArtifacts
from .validation import ValidationRunner

__all__ = [
    "MONTH_ORDER",
    "HotelDataProcessor",
    "NotebookEDAAnalyzer",
    "_count_model_complexity",
    "_one_hot_encoder",
    "_positive_probabilities",
    "_safe_float",
    "_slugify",
    "KerasTabularClassifier",
    "SHAPAnalyzer",
    "EvaluationMetrics",
    "ANNModel",
    "BaseHotelModel",
    "DecisionTreeModel",
    "KNNModel",
    "LSTMModel",
    "LogisticRegressionModel",
    "MODEL_REGISTRY",
    "RandomForestModel",
    "RNNModel",
    "SVMModel",
    "XGBoostModel",
    "ModelTester",
    "KMeansSegmenter",
    "ModelTrainer",
    "TerminalTrainingRunner",
    "TrainingArtifacts",
    "ValidationRunner",
]


## `hotel_app/ml/data.py`


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
import re
from typing import Any, Literal, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


MONTH_ORDER = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December",
]

FeaturePreset = Literal["honest", "high_score"]


def _one_hot_encoder(
    drop_first: bool = False,
    categories: Optional[Sequence[Sequence[Any]]] = None,
) -> OneHotEncoder:
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            categories=categories,
            drop="first" if drop_first else None,
            sparse_output=False,
            dtype=np.float32,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            categories=categories,
            drop="first" if drop_first else None,
            sparse=False,
            dtype=np.float32,
        )


def _positive_probabilities(model: Any, x_data: pd.DataFrame) -> Optional[np.ndarray]:
    """Return positive-class probabilities for any fitted classifier.

    Most classifiers in this project expose ``predict_proba`` directly.
    For score-based estimators that only expose ``decision_function``,
    such as a linear SVM before calibration, we convert scores with the
    logistic sigmoid function ``1 / (1 + exp(-score))`` so downstream
    metrics and dashboards can work with probability-style outputs.
    """
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(x_data)
        if probabilities.ndim == 2 and probabilities.shape[1] > 1:
            return probabilities[:, 1]
        return probabilities.ravel()
    if hasattr(model, "decision_function"):
        scores = model.decision_function(x_data)
        return 1 / (1 + np.exp(-scores))
    return None


def _slugify(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


def _safe_float(value: Any) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return float("nan")


def _count_model_complexity(estimator: Any) -> int:
    if hasattr(estimator, "estimator_"):
        return _count_model_complexity(estimator.estimator_)
    if hasattr(estimator, "best_estimator_"):
        return _count_model_complexity(estimator.best_estimator_)
    if hasattr(estimator, "tree_"):
        return int(getattr(estimator.tree_, "node_count", 0))
    if hasattr(estimator, "estimators_"):
        total = 0
        for inner in estimator.estimators_:
            if hasattr(inner, "tree_"):
                total += int(getattr(inner.tree_, "node_count", 0))
            else:
                total += 1
        return total
    if hasattr(estimator, "coef_"):
        coef = np.asarray(estimator.coef_)
        intercept = np.asarray(getattr(estimator, "intercept_", []))
        return int(coef.size + intercept.size)
    if hasattr(estimator, "coefs_"):
        total = sum(np.asarray(weights).size for weights in estimator.coefs_)
        total += sum(np.asarray(bias).size for bias in getattr(estimator, "intercepts_", []))
        return int(total)
    if hasattr(estimator, "support_vectors_"):
        return int(np.asarray(estimator.support_vectors_).shape[0])
    if hasattr(estimator, "n_features_in_"):
        return int(estimator.n_features_in_)
    return 0


@dataclass
class HotelDataProcessor:
    target_column: str = "is_canceled"
    reservation_target_column: str = "booking_status"
    dropped_low_signal_columns: Sequence[str] = field(default_factory=lambda: ("arrival_date_year",))
    dropped_behavior_columns: Sequence[str] = field(
        default_factory=lambda: (
            "country",
            "deposit_type",
            "required_car_parking_spaces",
            "assigned_room_type",
            "name",
            "email",
            "phone-number",
            "credit_card",
        )
    )
    leakage_columns: Sequence[str] = field(
        default_factory=lambda: ("reservation_status", "reservation_status_date")
    )
    reservation_guest_count_categories: Sequence[str] = field(
        default_factory=lambda: tuple(str(value) for value in range(1, 13))
    )

    def resolve_feature_preset(
        self,
        remove_leakage_features: bool = True,
        feature_preset: Optional[FeaturePreset] = None,
    ) -> FeaturePreset:
        if feature_preset is not None:
            return feature_preset
        return "honest" if remove_leakage_features else "high_score"

    @staticmethod
    def _snake_case(value: str) -> str:
        value = value.strip().lower()
        value = re.sub(r"[^a-z0-9]+", "_", value)
        return re.sub(r"_+", "_", value).strip("_")

    def _standardize_columns(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.copy()
        df.columns = [self._snake_case(str(column)) for column in df.columns]
        return df

    def detect_dataset(self, data: pd.DataFrame) -> str:
        columns = set(data.columns)
        if self.reservation_target_column in columns:
            return "reservation"
        if self.target_column in columns:
            return "hotel"
        raise KeyError("Could not find a supported target column in the dataset.")

    def resolve_target_column(self, data: pd.DataFrame) -> str:
        if self.target_column in data.columns:
            return self.target_column
        if self.reservation_target_column in data.columns:
            return self.reservation_target_column
        raise KeyError("Could not resolve the target column for the dataset.")

    def load_data(self, path: str = "hotel_bookings.csv") -> pd.DataFrame:
        return self._standardize_columns(pd.read_csv(path))

    def clean_data(self, data: pd.DataFrame) -> pd.DataFrame:
        df = self._standardize_columns(data)

        if "date_of_reservation" in df.columns:
            df["date_of_reservation"] = pd.to_datetime(
                df["date_of_reservation"],
                format="%m/%d/%Y",
                errors="coerce",
            )
            df = df.dropna(subset=["date_of_reservation"]).copy()

        for price_column in ("adr", "average_price"):
            if price_column in df.columns:
                prices = pd.to_numeric(df[price_column], errors="coerce").fillna(0)
                prices = prices.clip(lower=0)
                if price_column == "average_price" and self.reservation_target_column in df.columns and len(prices) > 10:
                    q1 = float(prices.quantile(0.25))
                    q3 = float(prices.quantile(0.75))
                    iqr = q3 - q1
                    lower_bound = max(0.0, q1 - 1.5 * iqr)
                    upper_bound = q3 + 1.5 * iqr
                    median_price = float(prices.median())
                    prices = prices.mask((prices < lower_bound) | (prices > upper_bound), median_price)
                elif len(prices) > 10:
                    lower_quantile = float(prices.quantile(0.01))
                    upper_quantile = float(prices.quantile(0.99))
                    prices = prices.clip(lower=lower_quantile, upper=upper_quantile)
                df[price_column] = prices
        guest_columns = [col for col in ("children", "adults", "babies") if col in df.columns]
        if guest_columns:
            df = df[df[guest_columns].sum(axis=1) > 0]
        object_columns = df.select_dtypes(include=["object"]).columns
        numeric_columns = df.select_dtypes(exclude=["object"]).columns
        df[object_columns] = df[object_columns].fillna("Unknown")
        df[numeric_columns] = df[numeric_columns].fillna(0)
        return df.reset_index(drop=True)

    def build_features(
        self,
        data: pd.DataFrame,
        remove_leakage_features: bool = True,
        feature_preset: Optional[FeaturePreset] = None,
    ) -> Tuple[pd.DataFrame, pd.Series]:
        df = self.clean_data(data)
        target_column = self.resolve_target_column(df)
        if target_column == self.reservation_target_column:
            y = df[target_column].astype(str).str.lower().eq("canceled").astype(int)
        else:
            y = pd.to_numeric(df[target_column], errors="coerce").fillna(0).astype(int)
        resolved_preset = self.resolve_feature_preset(
            remove_leakage_features=remove_leakage_features,
            feature_preset=feature_preset,
        )
        x_data = self.add_engineered_features(
            self.build_raw_prediction_inputs(
                df,
                remove_leakage_features=remove_leakage_features,
                feature_preset=resolved_preset,
            ),
            feature_preset=resolved_preset,
        )
        return x_data, y

    def build_raw_prediction_inputs(
        self,
        data: pd.DataFrame,
        remove_leakage_features: bool = True,
        feature_preset: Optional[FeaturePreset] = None,
    ) -> pd.DataFrame:
        df = self.clean_data(data)
        dataset_kind = self.detect_dataset(df)
        resolved_preset = self.resolve_feature_preset(
            remove_leakage_features=remove_leakage_features,
            feature_preset=feature_preset,
        )
        if dataset_kind == "reservation":
            return self._build_reservation_inputs(df)
        if resolved_preset == "high_score":
            return self._build_high_score_inputs(df)
        if resolved_preset == "honest":
            df = df.copy()
            if "country" in df.columns:
                country = df["country"].astype(str).fillna("Unknown")
                df["country_grouped"] = np.where(
                    country.eq("PRT"),
                    "PRT",
                    np.where(country.eq("GBR"), "GBR", "Other"),
                )
            if "required_car_parking_spaces" in df.columns:
                parking = pd.to_numeric(df["required_car_parking_spaces"], errors="coerce").fillna(0)
                df["needs_parking"] = (parking > 0).astype(int)
        drop_columns = list(self.dropped_low_signal_columns)
        drop_columns.extend(self.dropped_behavior_columns)
        if resolved_preset == "honest":
            drop_columns.extend(self.leakage_columns)
        drop_columns.append(self.target_column)
        year_columns = [col for col in df.columns if col.endswith("_year")]
        drop_columns.extend(year_columns)
        return df.drop(columns=[col for col in drop_columns if col in df.columns], errors="ignore")

    def _build_reservation_inputs(self, df: pd.DataFrame) -> pd.DataFrame:
        features = df.copy()
        lead_time_source = pd.to_numeric(features.get("lead_time"), errors="coerce").fillna(0)
        if {"number_of_adults", "number_of_children"}.issubset(features.columns):
            total_people = (
                pd.to_numeric(features["number_of_adults"], errors="coerce").fillna(0)
                + pd.to_numeric(features["number_of_children"], errors="coerce").fillna(0)
            ).astype(int)
            features["number_of_children_and_adults"] = total_people
        if {"number_of_weekend_nights", "number_of_week_nights"}.issubset(features.columns):
            total_nights = (
                pd.to_numeric(features["number_of_weekend_nights"], errors="coerce").fillna(0)
                + pd.to_numeric(features["number_of_week_nights"], errors="coerce").fillna(0)
            ).astype(int)
            features["number_of_total_nights"] = pd.cut(
                total_nights,
                bins=[-1, 0, 3, 7, 14, np.inf],
                labels=[0, 1, 2, 3, 4],
            ).astype("Int64").fillna(0).astype(int)
        if "lead_time" in features.columns:
            features["lead_time"] = pd.cut(
                lead_time_source,
                bins=[-1, 1, 7, 30, 365, np.inf],
                labels=[0, 1, 2, 3, 4],
            ).astype("Int64").fillna(0).astype(int)
        if {"p_c", "p_not_c", "repeated"}.issubset(features.columns):
            history_total = (
                pd.to_numeric(features["p_c"], errors="coerce").fillna(0)
                + pd.to_numeric(features["p_not_c"], errors="coerce").fillna(0)
            )
            repeated = pd.to_numeric(features["repeated"], errors="coerce").fillna(0).clip(lower=0, upper=1)
            features["cancellation_ratio"] = np.where(
                repeated > 0,
                pd.to_numeric(features["p_c"], errors="coerce").fillna(0) / history_total.replace(0, 1),
                0,
            )
            features["cancellation_ratio"] = features["cancellation_ratio"].round(2)
            features["first_time_visitor"] = 1 - repeated
        if "date_of_reservation" in features.columns:
            date_series = pd.to_datetime(features["date_of_reservation"], errors="coerce")
            features["day_name"] = date_series.dt.dayofweek.fillna(0).astype(int)
            features["month"] = date_series.dt.month.fillna(0).astype(int)
            features["year"] = date_series.dt.year.fillna(0).astype(int)

        drop_columns = [
            self.reservation_target_column,
            "booking_id",
            "date_of_reservation",
            "number_of_adults",
            "number_of_children",
            "number_of_weekend_nights",
            "number_of_week_nights",
            "repeated",
            "p_c",
            "p_not_c",
        ]
        return features.drop(columns=[col for col in drop_columns if col in features.columns], errors="ignore")

    def _build_high_score_inputs(self, df: pd.DataFrame) -> pd.DataFrame:
        features = df.copy()
        if {"arrival_date_year", "arrival_date_month", "arrival_date_day_of_month"}.issubset(features.columns):
            arrival_date = pd.to_datetime(
                {
                    "year": pd.to_numeric(features["arrival_date_year"], errors="coerce").fillna(2016).astype(int),
                    "month": features["arrival_date_month"].map({name: i + 1 for i, name in enumerate(MONTH_ORDER)}).fillna(1).astype(int),
                    "day": pd.to_numeric(features["arrival_date_day_of_month"], errors="coerce").fillna(1).astype(int),
                },
                errors="coerce",
            )
            features["arrival_date"] = arrival_date
        if {"assigned_room_type", "reserved_room_type"}.issubset(features.columns):
            features["change_in_room"] = (features["assigned_room_type"].astype(str) != features["reserved_room_type"].astype(str)).astype(int)
        if {"children", "babies"}.issubset(features.columns):
            features["offspring"] = (
                pd.to_numeric(features["children"], errors="coerce").fillna(0)
                + pd.to_numeric(features["babies"], errors="coerce").fillna(0)
            ).astype(int)
        if {"previous_cancellations", "previous_bookings_not_canceled"}.issubset(features.columns):
            features["total_bookings"] = (
                pd.to_numeric(features["previous_cancellations"], errors="coerce").fillna(0)
                + pd.to_numeric(features["previous_bookings_not_canceled"], errors="coerce").fillna(0)
            )
        if "country" in features.columns:
            country = features["country"].astype(str).fillna("Unknown")
            features["country_grouped"] = np.where(country.eq("PRT"), "PRT", np.where(country.eq("GBR"), "GBR", "Other"))
        if {"reservation_status_date", "arrival_date"}.issubset(features.columns):
            status_date = pd.to_datetime(features["reservation_status_date"], errors="coerce")
            arrival_date = pd.to_datetime(features["arrival_date"], errors="coerce")
            stay_duration = ((status_date - arrival_date) / np.timedelta64(1, "D")).fillna(-1)
            features["stay_duration"] = stay_duration.astype(int).clip(lower=-1)
        drop_columns = [
            self.target_column,
            "name",
            "email",
            "phone-number",
            "credit_card",
            "country",
            "arrival_date",
            "arrival_date_year",
            "reservation_status_date",
        ]
        return features.drop(columns=[col for col in drop_columns if col in features.columns], errors="ignore")

    def add_engineered_features(
        self,
        x_data: pd.DataFrame,
        feature_preset: Optional[FeaturePreset] = None,
    ) -> pd.DataFrame:
        resolved_preset = feature_preset or "honest"
        features = x_data.copy()
        if self._is_reservation_feature_frame(features):
            if "lead_time" in features.columns:
                features["lead_time"] = pd.to_numeric(features["lead_time"], errors="coerce").fillna(0).astype(int)
            if "number_of_total_nights" in features.columns:
                features["number_of_total_nights"] = (
                    pd.to_numeric(features["number_of_total_nights"], errors="coerce").fillna(0).astype(int)
                )
            if "day_name" in features.columns:
                features["day_name"] = pd.to_numeric(features["day_name"], errors="coerce").fillna(0).astype(int)
            if "number_of_children_and_adults" in features.columns:
                features["number_of_children_and_adults"] = (
                    pd.to_numeric(features["number_of_children_and_adults"], errors="coerce")
                    .fillna(0)
                    .astype(int)
                )
            if "first_time_visitor" in features.columns:
                features["first_time_visitor"] = (
                    pd.to_numeric(features["first_time_visitor"], errors="coerce").fillna(0).clip(0, 1).astype(int)
                )
            if "cancellation_ratio" in features.columns:
                features["cancellation_ratio"] = (
                    pd.to_numeric(features["cancellation_ratio"], errors="coerce").fillna(0).clip(0, 1).round(2)
                )
            return features
        if {"average_price", "number_of_children_and_adults"}.issubset(features.columns):
            features["average_price_per_guest"] = (
                pd.to_numeric(features["average_price"], errors="coerce").fillna(0)
                / pd.to_numeric(features["number_of_children_and_adults"], errors="coerce").fillna(0).replace(0, 1)
            ).fillna(0)
            total_people = pd.to_numeric(features["number_of_children_and_adults"], errors="coerce").fillna(0)
            features["guest_party_type"] = np.select(
                [total_people <= 1, total_people == 2, total_people <= 4],
                ["Solo", "Couple", "Small Group"],
                default="Large Group",
            )
        if {"average_price", "number_of_total_nights"}.issubset(features.columns):
            features["total_stay_value"] = (
                pd.to_numeric(features["average_price"], errors="coerce").fillna(0)
                * pd.to_numeric(features["number_of_total_nights"], errors="coerce").fillna(0)
            )
        if "lead_time" in features.columns and "lead_time_band" not in features.columns:
            lead_time_value = pd.to_numeric(features.get("lead_time_raw", features["lead_time"]), errors="coerce").fillna(0)
            features["lead_time_band"] = lead_time_value.map(
                lambda value: "Same Day" if value <= 1 else "Short Notice" if value <= 7 else "Medium Term" if value <= 30 else "Long Term" if value <= 365 else "Very Long Term"
            )
            features["lead_time_bucket_code"] = lead_time_value.map(
                lambda value: 0 if value <= 1 else 1 if value <= 7 else 2 if value <= 30 else 3 if value <= 365 else 4
            )
        if "number_of_total_nights" in features.columns and "stay_length_bucket" not in features.columns:
            total_nights_bucket = pd.to_numeric(features["number_of_total_nights"], errors="coerce").fillna(0)
            features["stay_length_bucket"] = total_nights_bucket.map(
                lambda value: "Day Use" if value == 0 else "Short Stay" if value <= 3 else "Week Stay" if value <= 7 else "Two Weeks Stay" if value <= 14 else "Long Stay"
            )
            features["stay_length_bucket_code"] = total_nights_bucket.map(
                lambda value: 0 if value == 0 else 1 if value <= 3 else 2 if value <= 7 else 3 if value <= 14 else 4
            )
        if {"special_requests", "number_of_total_nights"}.issubset(features.columns):
            features["special_requests_per_night"] = (
                pd.to_numeric(features["special_requests"], errors="coerce").fillna(0)
                / pd.to_numeric(features["number_of_total_nights"], errors="coerce").fillna(0).replace(0, 1)
            ).fillna(0)
        if {"special_requests", "number_of_children_and_adults"}.issubset(features.columns):
            features["special_requests_per_guest"] = (
                pd.to_numeric(features["special_requests"], errors="coerce").fillna(0)
                / pd.to_numeric(features["number_of_children_and_adults"], errors="coerce").fillna(0).replace(0, 1)
            ).fillna(0)
        if "agent" in features.columns:
            agent_numeric = pd.to_numeric(features["agent"], errors="coerce").fillna(0)
            features["has_agent"] = (agent_numeric > 0).astype(int)
            if resolved_preset == "honest":
                features = features.drop(columns=["agent"])
        if "company" in features.columns:
            company_numeric = pd.to_numeric(features["company"], errors="coerce").fillna(0)
            features["has_company"] = (company_numeric > 0).astype(int)
            if resolved_preset == "honest":
                features = features.drop(columns=["company"])
        if {"stays_in_weekend_nights", "stays_in_week_nights"}.issubset(features.columns):
            features["total_nights"] = (
                features["stays_in_weekend_nights"] + features["stays_in_week_nights"]
            )
            features["weekend_share"] = (
                features["stays_in_weekend_nights"] / features["total_nights"].replace(0, 1)
            ).fillna(0)
            features["stay_length_bucket"] = pd.cut(
                features["total_nights"],
                bins=[-1, 0, 3, 7, 14, np.inf],
                labels=["Day Use", "Short Stay", "Week Stay", "Two Weeks Stay", "Long Stay"],
            ).astype(str)
        if {"adults", "children", "babies"}.issubset(features.columns):
            features["total_guests"] = features["adults"] + features["children"] + features["babies"]
            features["family_booking"] = ((features["children"] + features["babies"]) > 0).astype(int)
            features["adults_only"] = ((features["adults"] > 0) & (features["children"] + features["babies"] == 0)).astype(int)
            features["guest_party_type"] = np.select(
                [
                    features["total_guests"] <= 1,
                    features["total_guests"] == 2,
                    features["total_guests"] <= 4,
                ],
                ["Solo", "Couple", "Small Group"],
                default="Large Group",
            )
        if "is_repeated_guest" in features.columns:
            repeated_guest = pd.to_numeric(features["is_repeated_guest"], errors="coerce").fillna(0).clip(lower=0, upper=1)
            features["first_time_visitor"] = 1 - repeated_guest
        if {"previous_cancellations", "previous_bookings_not_canceled"}.issubset(features.columns):
            previous_total = features["previous_cancellations"] + features["previous_bookings_not_canceled"]
            features["total_bookings"] = previous_total
            features["previous_cancel_rate"] = (
                features["previous_cancellations"] / previous_total.replace(0, 1)
            ).fillna(0)
            features["has_booking_history"] = (previous_total > 0).astype(int)
        if {"booking_changes", "lead_time"}.issubset(features.columns):
            features["changes_per_lead_day"] = (
                features["booking_changes"] / features["lead_time"].replace(0, 1)
            ).fillna(0)
        if {"booking_changes", "total_nights"}.issubset(features.columns):
            features["changes_per_night"] = (
                features["booking_changes"] / features["total_nights"].replace(0, 1)
            ).fillna(0)
        if {"total_of_special_requests", "total_nights"}.issubset(features.columns):
            features["requests_per_night"] = (
                features["total_of_special_requests"] / features["total_nights"].replace(0, 1)
            ).fillna(0)
        if {"total_of_special_requests", "total_guests"}.issubset(features.columns):
            features["requests_per_guest"] = (
                features["total_of_special_requests"] / features["total_guests"].replace(0, 1)
            ).fillna(0)
        if {"adr", "total_guests"}.issubset(features.columns):
            features["adr_per_guest"] = (
                features["adr"] / features["total_guests"].replace(0, 1)
            ).fillna(0)
        if {"adr", "total_nights"}.issubset(features.columns):
            features["booking_value"] = features["adr"] * features["total_nights"]
            features["adr_per_night_log"] = np.log1p(features["adr"].clip(lower=0))
        if {"booking_value", "total_guests", "total_nights"}.issubset(features.columns):
            features["value_per_guest_night"] = (
                features["booking_value"]
                / (features["total_guests"].replace(0, 1) * features["total_nights"].replace(0, 1))
            ).fillna(0)
        if {"total_guests", "total_nights"}.issubset(features.columns):
            features["guests_per_night"] = (
                features["total_guests"] / features["total_nights"].replace(0, 1)
            ).fillna(0)
        if {"lead_time", "total_guests"}.issubset(features.columns):
            features["lead_time_per_guest"] = (
                features["lead_time"] / features["total_guests"].replace(0, 1)
            ).fillna(0)
        if {"lead_time", "total_nights"}.issubset(features.columns):
            features["lead_time_per_night"] = (
                features["lead_time"] / features["total_nights"].replace(0, 1)
            ).fillna(0)
        if "lead_time" in features.columns:
            features["lead_time_log"] = np.log1p(features["lead_time"])
            features["lead_time_bucket"] = pd.cut(
                features["lead_time"],
                bins=[-1, 7, 30, 90, 180, np.inf],
                labels=["Last Minute", "Short", "Medium", "Long", "Very Long"],
            ).astype(str)
            features["lead_time_band"] = pd.cut(
                features["lead_time"],
                bins=[-1, 1, 7, 30, 365, np.inf],
                labels=["Same Day", "Short Notice", "Medium Term", "Long Term", "Very Long Term"],
            ).astype(str)
        if "days_in_waiting_list" in features.columns:
            features["waiting_list_log"] = np.log1p(features["days_in_waiting_list"].clip(lower=0))
        if "arrival_date_month" in features.columns:
            month_index = features["arrival_date_month"].map({name: i + 1 for i, name in enumerate(MONTH_ORDER)}).fillna(0)
            features["arrival_month_index"] = month_index
            features["arrival_month_sin"] = np.sin(2 * np.pi * month_index / 12.0)
            features["arrival_month_cos"] = np.cos(2 * np.pi * month_index / 12.0)
        if "arrival_date_week_number" in features.columns:
            week = pd.to_numeric(features["arrival_date_week_number"], errors="coerce").fillna(0)
            features["arrival_week_sin"] = np.sin(2 * np.pi * week / 53.0)
            features["arrival_week_cos"] = np.cos(2 * np.pi * week / 53.0)
        return features

    def build_preprocessor(self, x_data: pd.DataFrame) -> ColumnTransformer:
        if self._is_reservation_feature_frame(x_data):
            categorical_columns = list(x_data.select_dtypes(include=["object", "category"]).columns)
            forced_categorical = [column for column in ("number_of_children_and_adults",) if column in x_data.columns]
            categorical_columns = list(dict.fromkeys([*categorical_columns, *forced_categorical]))
            numeric_columns = [column for column in x_data.columns if column not in categorical_columns]
            scaled_numeric_columns = [column for column in ("average_price",) if column in numeric_columns]
            passthrough_numeric_columns = [
                column for column in numeric_columns if column not in scaled_numeric_columns
            ]
            transformers = []
            if scaled_numeric_columns:
                transformers.append(
                    (
                        "scaled_numeric",
                        Pipeline(
                            steps=[
                                ("imputer", SimpleImputer(strategy="median")),
                                ("scaler", StandardScaler()),
                            ]
                        ),
                        scaled_numeric_columns,
                    )
                )
            if passthrough_numeric_columns:
                transformers.append(
                    (
                        "numeric",
                        Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
                        passthrough_numeric_columns,
                    )
                )
            if categorical_columns:
                reservation_categories = {
                    "type_of_meal": ["Meal Plan 1", "Meal Plan 2", "Meal Plan 3", "Not Selected"],
                    "room_type": [f"Room_Type {index}" for index in range(1, 8)],
                    "market_segment_type": ["Aviation", "Complementary", "Corporate", "Offline", "Online"],
                    "number_of_children_and_adults": list(range(1, 13)),
                }
                category_lists = [
                    reservation_categories.get(
                        column,
                        sorted(pd.Series(x_data[column]).dropna().unique().tolist()),
                    )
                    for column in categorical_columns
                ]
                transformers.append(
                    (
                        "categorical",
                        Pipeline(
                            steps=[
                                ("imputer", SimpleImputer(strategy="most_frequent")),
                                ("encoder", _one_hot_encoder(drop_first=True, categories=category_lists)),
                            ]
                        ),
                        categorical_columns,
                    )
                )
            return ColumnTransformer(
                transformers=transformers,
                remainder="drop",
                verbose_feature_names_out=False,
            )
        categorical_columns = list(x_data.select_dtypes(include=["object", "category"]).columns)
        numeric_columns = [col for col in x_data.columns if col not in categorical_columns]
        numeric_pipeline = Pipeline(
            steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
        )
        categorical_pipeline = Pipeline(
            steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", _one_hot_encoder())]
        )
        return ColumnTransformer(
            transformers=[
                ("numeric", numeric_pipeline, numeric_columns),
                ("categorical", categorical_pipeline, categorical_columns),
            ],
            remainder="drop",
            verbose_feature_names_out=False,
        )

    @staticmethod
    def _is_reservation_feature_frame(x_data: pd.DataFrame) -> bool:
        reservation_markers = {
            "type_of_meal",
            "car_parking_space",
            "room_type",
            "market_segment_type",
            "average_price",
            "special_requests",
        }
        return len(reservation_markers.intersection(x_data.columns)) >= 4


class NotebookEDAAnalyzer:
    def __init__(self, data: pd.DataFrame, processor: Optional[HotelDataProcessor] = None) -> None:
        self.raw_data = data.copy()
        self.processor = processor or HotelDataProcessor()
        self.clean_data = self.processor.clean_data(data)

    def preview(self, rows: int = 5) -> pd.DataFrame:
        return self.raw_data.head(rows)


## `hotel_app/ml/deep.py`


In [ ]:
from __future__ import annotations

from typing import Any, Sequence
import importlib

import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.class_weight import compute_class_weight


class KerasTabularClassifier(BaseEstimator, ClassifierMixin):
    def __init__(
        self,
        model_type: str = "ann",
        epochs: int = 20,
        batch_size: int = 256,
        learning_rate: float = 0.001,
        random_state: int = 42,
        verbose: int = 0,
    ) -> None:
        self.model_type = model_type
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.verbose = verbose

    def _build_model(self, n_features: int) -> Any:
        try:
            tf = importlib.import_module("tensorflow")
        except ImportError as exc:
            raise ImportError(
                "TensorFlow is required for ANNModel, RNNModel, and LSTMModel. "
                "Install it with `pip install tensorflow` or remove deep models from the run."
            ) from exc
        tf.random.set_seed(self.random_state)
        regularizer = tf.keras.regularizers.l2(1e-4)
        model = tf.keras.Sequential()
        if self.model_type == "rnn":
            model.add(tf.keras.layers.Input(shape=(n_features, 1)))
            model.add(
                tf.keras.layers.Bidirectional(
                    tf.keras.layers.SimpleRNN(
                        40,
                        activation="tanh",
                        dropout=0.1,
                        recurrent_dropout=0.1,
                    )
                )
            )
            model.add(tf.keras.layers.BatchNormalization())
            model.add(tf.keras.layers.Dropout(0.25))
            model.add(tf.keras.layers.Dense(24, activation="relu", kernel_regularizer=regularizer))
        elif self.model_type == "lstm":
            model.add(tf.keras.layers.Input(shape=(n_features, 1)))
            model.add(
                tf.keras.layers.Bidirectional(
                    tf.keras.layers.LSTM(
                        32,
                        activation="tanh",
                        dropout=0.1,
                        recurrent_dropout=0.1,
                    )
                )
            )
            model.add(tf.keras.layers.BatchNormalization())
            model.add(tf.keras.layers.Dropout(0.25))
            model.add(tf.keras.layers.Dense(24, activation="relu", kernel_regularizer=regularizer))
        else:
            model.add(tf.keras.layers.Input(shape=(n_features,)))
            model.add(tf.keras.layers.Dense(128, activation="relu", kernel_regularizer=regularizer))
            model.add(tf.keras.layers.BatchNormalization())
            model.add(tf.keras.layers.Dropout(0.25))
            model.add(tf.keras.layers.Dense(64, activation="relu", kernel_regularizer=regularizer))
            model.add(tf.keras.layers.BatchNormalization())
            model.add(tf.keras.layers.Dropout(0.15))
            model.add(tf.keras.layers.Dense(24, activation="relu", kernel_regularizer=regularizer))
        model.add(tf.keras.layers.Dense(1, activation="sigmoid"))
        optimizer = tf.keras.optimizers.Adam(learning_rate=self.learning_rate, clipnorm=1.0)
        model.compile(
            optimizer=optimizer,
            loss="binary_crossentropy",
            metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
        )
        return model

    def _reshape(self, x_data: Any) -> np.ndarray:
        array = np.asarray(x_data, dtype=np.float32)
        if self.model_type in {"rnn", "lstm"}:
            return array.reshape(array.shape[0], array.shape[1], 1)
        return array

    def fit(self, x_data: Any, y_data: Sequence[int]) -> "KerasTabularClassifier":
        if hasattr(x_data, "toarray"):
            x_data = x_data.toarray()
        array = np.asarray(x_data, dtype=np.float32)
        y_array = np.asarray(y_data, dtype=np.int32)
        self.classes_ = np.array([0, 1])
        self.n_features_in_ = array.shape[1]
        self.model_ = self._build_model(self.n_features_in_)
        tf = importlib.import_module("tensorflow")
        present_classes = np.unique(y_array)
        class_weights = compute_class_weight(class_weight="balanced", classes=present_classes, y=y_array)
        class_weight_map = {int(label): float(weight) for label, weight in zip(present_classes, class_weights)}
        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_auc",
                mode="max",
                patience=6,
                restore_best_weights=True,
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=3,
                min_lr=1e-5,
            ),
        ]
        self.history_ = self.model_.fit(
            self._reshape(array),
            y_array.astype(np.float32),
            epochs=self.epochs,
            batch_size=self.batch_size,
            validation_split=0.1,
            callbacks=callbacks,
            class_weight=class_weight_map,
            verbose=self.verbose,
        )
        return self

    def predict_proba(self, x_data: Any) -> np.ndarray:
        if hasattr(x_data, "toarray"):
            x_data = x_data.toarray()
        probabilities = self.model_.predict(self._reshape(x_data), verbose=0).ravel()
        return np.vstack([1 - probabilities, probabilities]).T

    def predict(self, x_data: Any) -> np.ndarray:
        return (self.predict_proba(x_data)[:, 1] >= 0.5).astype(int)


## `hotel_app/ml/explainability.py`


In [ ]:
from __future__ import annotations

from typing import Any

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline


class SHAPAnalyzer:
    def __init__(self, random_state: int = 42) -> None:
        self.random_state = random_state

    def explain(
        self,
        model: Pipeline,
        x_background: pd.DataFrame,
        x_explain: pd.DataFrame,
        max_background: int = 100,
    ) -> Any:
        import shap
        from scipy import sparse

        background = x_background.sample(min(max_background, len(x_background)), random_state=self.random_state)
        preprocessor = model.named_steps["preprocessor"]
        estimator = model.named_steps["model"]
        background_values = preprocessor.transform(background)
        explain_values = preprocessor.transform(x_explain)
        if sparse.issparse(background_values):
            background_values = background_values.toarray()
        if sparse.issparse(explain_values):
            explain_values = explain_values.toarray()
        try:
            feature_names = preprocessor.get_feature_names_out()
        except Exception:
            feature_names = [f"feature_{index}" for index in range(background_values.shape[1])]
        background_frame = pd.DataFrame(background_values, columns=feature_names)
        explain_frame = pd.DataFrame(explain_values, columns=feature_names)

        def predict_fn(values: np.ndarray) -> np.ndarray:
            return estimator.predict_proba(np.asarray(values, dtype=np.float32))[:, 1]

        explainer = shap.Explainer(predict_fn, background_frame)
        return explainer(explain_frame)

    def summary_plot(self, shap_values: Any, max_display: int = 15) -> Any:
        import matplotlib.pyplot as plt
        import shap

        shap.summary_plot(shap_values, show=False, max_display=max_display)
        figure = plt.gcf()
        plt.tight_layout()
        return figure


## `hotel_app/ml/metrics.py`


In [ ]:
from __future__ import annotations

from typing import Any, Dict, Optional, Sequence

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)


class EvaluationMetrics:
    metric_names = (
        "accuracy",
        "precision",
        "recall",
        "f1",
        "balanced_accuracy",
        "roc_auc",
        "average_precision",
        "brier_score",
        "log_loss",
        "mcc",
    )

    @staticmethod
    def evaluate(
        y_true: Sequence[int],
        y_pred: Sequence[int],
        y_score: Optional[Sequence[float]] = None,
    ) -> Dict[str, float]:
        metrics = {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
            "mcc": matthews_corrcoef(y_true, y_pred),
        }
        if y_score is not None:
            try:
                metrics["roc_auc"] = roc_auc_score(y_true, y_score)
            except ValueError:
                metrics["roc_auc"] = np.nan
            try:
                metrics["average_precision"] = average_precision_score(y_true, y_score)
            except ValueError:
                metrics["average_precision"] = np.nan
            try:
                metrics["brier_score"] = brier_score_loss(y_true, y_score)
            except ValueError:
                metrics["brier_score"] = np.nan
            try:
                metrics["log_loss"] = log_loss(y_true, y_score)
            except ValueError:
                metrics["log_loss"] = np.nan
        else:
            metrics["roc_auc"] = np.nan
            metrics["average_precision"] = np.nan
            metrics["brier_score"] = np.nan
            metrics["log_loss"] = np.nan
        return metrics

    @staticmethod
    def report(
        y_true: Sequence[int],
        y_pred: Sequence[int],
        y_score: Optional[Sequence[float]] = None,
    ) -> Dict[str, Any]:
        return {
            "metrics": EvaluationMetrics.evaluate(y_true, y_pred, y_score),
            "confusion_matrix": confusion_matrix(y_true, y_pred),
            "classification_report": classification_report(
                y_true, y_pred, output_dict=True, zero_division=0
            ),
        }


## `hotel_app/ml/testing.py`


In [ ]:
from __future__ import annotations

from typing import Any, Dict, Tuple

import pandas as pd
from sklearn.pipeline import Pipeline

from .data import _positive_probabilities
from .metrics import EvaluationMetrics


class ModelTester:
    @staticmethod
    def _prefix_metrics(metrics: Dict[str, Any], prefix: str) -> Dict[str, Any]:
        return {f"{prefix}_{key}": value for key, value in metrics.items()}

    def test_model(
        self,
        name: str,
        model: Pipeline,
        x_train: pd.DataFrame,
        y_train: pd.Series,
        x_test: pd.DataFrame,
        y_test: pd.Series,
    ) -> Dict[str, Any]:
        train_pred = model.predict(x_train)
        train_score = _positive_probabilities(model, x_train)
        y_pred = model.predict(x_test)
        y_score = _positive_probabilities(model, x_test)
        result = EvaluationMetrics.report(y_test, y_pred, y_score)
        result["train_metrics"] = EvaluationMetrics.evaluate(y_train, train_pred, train_score)
        result["metrics"].update(self._prefix_metrics(result["train_metrics"], "train"))
        result["model"] = name
        return result

    def test_many(
        self,
        models: Dict[str, Pipeline],
        x_train: pd.DataFrame,
        y_train: pd.Series,
        x_test: pd.DataFrame,
        y_test: pd.Series,
    ) -> Tuple[pd.DataFrame, Dict[str, Dict[str, Any]]]:
        details = {
            name: self.test_model(name, model, x_train, y_train, x_test, y_test)
            for name, model in models.items()
        }
        summary = pd.DataFrame([dict(model=name, **details[name]["metrics"]) for name in details]).sort_values(
            "f1", ascending=False
        )
        return summary, details


## `hotel_app/ml/training.py`


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
import json
import importlib
from pathlib import Path
import sys
import time
from typing import Any, Dict, Iterable, List, Optional, Sequence

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from .data import HotelDataProcessor, _count_model_complexity, _positive_probabilities, _safe_float, _slugify
from .explainability import SHAPAnalyzer
from .metrics import EvaluationMetrics
from .models import (
    ANNModel,
    BaseHotelModel,
    DecisionTreeModel,
    KNNModel,
    LSTMModel,
    LogisticRegressionModel,
    RandomForestModel,
    RNNModel,
    SVMModel,
    XGBoostModel,
)


class ModelTrainer:
    def __init__(
        self,
        processor: Optional[HotelDataProcessor] = None,
        random_state: int = 42,
        test_size: float = 0.2,
    ) -> None:
        self.processor = processor or HotelDataProcessor()
        self.random_state = random_state
        self.test_size = test_size

    def prepare_data(
        self,
        data_path: str = "hotel_bookings.csv",
        sample_size: Optional[int] = None,
        remove_leakage_features: bool = True,
        feature_preset: Optional[str] = None,
    ):
        data = self.processor.load_data(data_path)
        if sample_size and sample_size < len(data):
            data = data.sample(sample_size, random_state=self.random_state)
        return self.processor.build_features(
            data,
            remove_leakage_features=remove_leakage_features,
            feature_preset=feature_preset,
        )

    def split_data(self, x_data: pd.DataFrame, y_data: pd.Series):
        stratify_target = None if self.processor._is_reservation_feature_frame(x_data) else y_data
        return train_test_split(
            x_data,
            y_data,
            test_size=self.test_size,
            stratify=stratify_target,
            random_state=self.random_state,
        )

    def train_model(self, model_spec: BaseHotelModel, x_train: pd.DataFrame, y_train: pd.Series) -> Pipeline:
        preprocessor = self.processor.build_preprocessor(x_train)
        pipeline = model_spec.build_pipeline(preprocessor)
        return pipeline.fit(x_train, y_train)

    def retrain_from_benchmark(self, benchmark_model: Pipeline, x_data: pd.DataFrame, y_data: pd.Series) -> Pipeline:
        preprocessor = clone(benchmark_model.named_steps["preprocessor"])
        benchmark_estimator = benchmark_model.named_steps["model"]
        if hasattr(benchmark_estimator, "best_params_") and hasattr(benchmark_estimator, "estimator"):
            final_estimator = clone(benchmark_estimator.estimator)
            final_estimator.set_params(**benchmark_estimator.best_params_)
        elif hasattr(benchmark_estimator, "best_estimator_"):
            final_estimator = clone(benchmark_estimator.best_estimator_)
        else:
            final_estimator = clone(benchmark_estimator)
        pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", final_estimator)])
        return pipeline.fit(x_data, y_data)

    def train_many(self, model_specs: Iterable[BaseHotelModel], x_train: pd.DataFrame, y_train: pd.Series):
        return {model.name: self.train_model(model, x_train, y_train) for model in model_specs}

    def resample_training_data(self, x_train: pd.DataFrame, y_train: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
        try:
            from imblearn.over_sampling import SMOTE, SMOTENC
        except ImportError:
            return x_train, y_train

        categorical_columns = list(x_train.select_dtypes(include=["object", "category"]).columns)
        if categorical_columns:
            sampler = SMOTENC(
                categorical_features=[x_train.columns.get_loc(column) for column in categorical_columns],
                random_state=self.random_state,
            )
        else:
            sampler = SMOTE(random_state=self.random_state)

        x_resampled, y_resampled = sampler.fit_resample(x_train, y_train)
        if not isinstance(x_resampled, pd.DataFrame):
            x_resampled = pd.DataFrame(x_resampled, columns=x_train.columns)
        for column in x_train.columns:
            if pd.api.types.is_numeric_dtype(x_train[column]):
                replacement = pd.to_numeric(x_train[column], errors="coerce").median()
                x_resampled[column] = pd.to_numeric(x_resampled[column], errors="coerce").fillna(replacement)
            else:
                x_resampled[column] = x_resampled[column].astype(str)
        return x_resampled.reset_index(drop=True), pd.Series(y_resampled, name=y_train.name)

    def k_fold_cross_validate(
        self,
        model_specs: Iterable[BaseHotelModel],
        x_data: pd.DataFrame,
        y_data: pd.Series,
        n_splits: int = 5,
    ) -> pd.DataFrame:
        if n_splits < 2:
            return pd.DataFrame()
        rows: List[Dict[str, Any]] = []
        splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=self.random_state)
        for model_spec in model_specs:
            fold_metrics: List[Dict[str, float]] = []
            for fold, (train_index, valid_index) in enumerate(splitter.split(x_data, y_data), start=1):
                x_train, x_valid = x_data.iloc[train_index], x_data.iloc[valid_index]
                y_train, y_valid = y_data.iloc[train_index], y_data.iloc[valid_index]
                fitted_model = self.train_model(model_spec, x_train, y_train)
                y_pred = fitted_model.predict(x_valid)
                y_score = _positive_probabilities(fitted_model, x_valid)
                metrics = EvaluationMetrics.evaluate(y_valid, y_pred, y_score)
                metrics.update({"model": model_spec.name, "fold": fold})
                fold_metrics.append(metrics)
                rows.append(metrics)
            average_metrics = pd.DataFrame(fold_metrics).mean(numeric_only=True).to_dict()
            average_metrics.update({"model": model_spec.name, "fold": "mean"})
            rows.append(average_metrics)
        return pd.DataFrame(rows)


class KMeansSegmenter:
    def __init__(self, random_state: int = 42, n_clusters: int = 4) -> None:
        self.random_state = random_state
        self.n_clusters = n_clusters

    def fit(self, data: pd.DataFrame) -> Dict[str, Any]:
        numeric_features = [
            column
            for column in (
                "lead_time",
                "adr",
                "average_price",
                "total_nights",
                "number_of_total_nights",
                "total_guests",
                "number_of_children_and_adults",
                "special_requests",
                "previous_cancellations",
                "cancellation_ratio",
            )
            if column in data.columns
        ]
        if not numeric_features:
            projection_frame = pd.DataFrame({"pc1": [], "pc2": [], "segment": []})
            return {
                "model": None,
                "summary": pd.DataFrame(),
                "projection": projection_frame,
                "feature_columns": [],
            }
        working = data[numeric_features].copy()
        scaler = StandardScaler()
        scaled = scaler.fit_transform(working)
        cluster_count = min(self.n_clusters, max(len(working), 1))
        model = KMeans(n_clusters=cluster_count, random_state=self.random_state, n_init=20)
        labels = model.fit_predict(scaled)
        enriched = working.copy()
        enriched["segment"] = labels
        if scaled.shape[1] >= 2:
            pca = PCA(n_components=2, random_state=self.random_state)
            projection = pca.fit_transform(scaled)
            projection_frame = pd.DataFrame({"pc1": projection[:, 0], "pc2": projection[:, 1], "segment": labels})
        else:
            projection_frame = pd.DataFrame({"pc1": scaled[:, 0], "pc2": np.zeros(len(scaled)), "segment": labels})
        segment_summary = enriched.groupby("segment").mean(numeric_only=True).round(2).reset_index()
        return {
            "model": model,
            "summary": segment_summary,
            "projection": projection_frame,
            "feature_columns": numeric_features,
        }


class TrainingArtifacts:
    def __init__(self, output_dir: str | Path) -> None:
        self.root = Path(output_dir)
        self.models_dir = self.root / "models"
        self.plots_dir = self.root / "plots"
        self.reports_dir = self.root / "reports"
        for directory in (self.root, self.models_dir, self.plots_dir, self.reports_dir):
            directory.mkdir(parents=True, exist_ok=True)

    def save_model(self, name: str, model: Pipeline) -> Path:
        path = self.models_dir / f"{_slugify(name)}.joblib"
        joblib.dump(model, path, compress=3)
        return path

    def save_dataframe(self, name: str, frame: pd.DataFrame) -> Path:
        path = self.reports_dir / name
        if path.suffix.lower() == ".json":
            frame.to_json(path, orient="records", indent=2)
        else:
            frame.to_csv(path, index=False)
        return path

    def save_json(self, name: str, payload: Dict[str, Any]) -> Path:
        path = self.reports_dir / name
        with path.open("w", encoding="utf-8") as handle:
            json.dump(payload, handle, indent=2, default=self._json_default)
        return path

    @staticmethod
    def _json_default(value: Any) -> Any:
        if isinstance(value, (np.integer, np.floating)):
            return value.item()
        if isinstance(value, np.ndarray):
            return value.tolist()
        return value


class TerminalTrainingRunner:
    def __init__(self, trainer: Optional[ModelTrainer] = None, random_state: int = 42) -> None:
        self.trainer = trainer or ModelTrainer(random_state=random_state, test_size=0.2)
        self.random_state = random_state
        self.processor = self.trainer.processor

    def default_models(
        self,
        ann_epochs: int = 250,
        rnn_epochs: int = 10,
        lstm_epochs: int = 10,
        selected_models: Optional[Sequence[str]] = None,
    ) -> List[BaseHotelModel]:
        default_order = [
            "Logistic Regression",
            "KNN",
            "Decision Tree",
            "Random Forest",
            "SVM",
            "XGBoost",
            "ANN",
            "LSTM",
            "RNN",
        ]
        model_names = list(selected_models) if selected_models else default_order
        models: List[BaseHotelModel] = []
        tensorflow_available = False
        try:
            importlib.import_module("tensorflow")
            tensorflow_available = True
        except ImportError:
            tensorflow_available = False

        for model_name in model_names:
            if model_name == "ANN":
                models.append(ANNModel(epochs=ann_epochs))
            elif model_name == "Logistic Regression":
                models.append(LogisticRegressionModel())
            elif model_name == "KNN":
                models.append(KNNModel())
            elif model_name == "Decision Tree":
                models.append(DecisionTreeModel())
            elif model_name == "Random Forest":
                models.append(RandomForestModel())
            elif model_name == "SVM":
                models.append(SVMModel())
            elif model_name == "XGBoost":
                models.append(XGBoostModel())
            elif model_name == "LSTM":
                if tensorflow_available:
                    models.append(LSTMModel(epochs=lstm_epochs))
            elif model_name == "RNN":
                if tensorflow_available:
                    models.append(RNNModel(epochs=rnn_epochs))
            else:
                raise ValueError(f"Unknown model requested: {model_name}")
        return models

    def run(
        self,
        data_path: str,
        output_dir: str = "artifacts",
        cv_folds: int = 5,
        ann_epochs: int = 250,
        rnn_epochs: int = 10,
        lstm_epochs: int = 10,
        shap_rows: int = 250,
        selected_models: Optional[Sequence[str]] = None,
        remove_leakage_features: bool = True,
        feature_preset: Optional[str] = None,
    ) -> Dict[str, Any]:
        from .testing import ModelTester

        pipeline_start = time.perf_counter()
        tester = ModelTester()
        artifacts = TrainingArtifacts(output_dir)
        raw_data = self.processor.load_data(data_path)
        dataset_kind = self.processor.detect_dataset(raw_data)
        resolved_preset = self.processor.resolve_feature_preset(
            remove_leakage_features=remove_leakage_features,
            feature_preset=feature_preset,
        )
        prediction_inputs = self.processor.build_raw_prediction_inputs(
            raw_data,
            remove_leakage_features=remove_leakage_features,
            feature_preset=resolved_preset,
        )
        x_data, y_data = self.processor.build_features(
            raw_data,
            remove_leakage_features=remove_leakage_features,
            feature_preset=resolved_preset,
        )
        x_train, x_test, y_train, y_test = self.trainer.split_data(x_data, y_data)
        x_fit, y_fit = x_train, y_train
        balance_strategy = "none"
        if dataset_kind == "reservation":
            try:
                x_fit, y_fit = self.trainer.resample_training_data(x_train, y_train)
                if len(x_fit) > len(x_train):
                    balance_strategy = "smote"
            except Exception as exc:
                balance_strategy = f"smote_failed: {exc}"
        models = self.default_models(
            ann_epochs=ann_epochs,
            rnn_epochs=rnn_epochs,
            lstm_epochs=lstm_epochs,
            selected_models=selected_models,
        )

        benchmark_rows: List[Dict[str, Any]] = []
        benchmark_rows_by_name: Dict[str, Dict[str, Any]] = {}  # keyed for later timing update
        details: Dict[str, Dict[str, Any]] = {}
        trained_models: Dict[str, Pipeline] = {}
        skipped_models: Dict[str, str] = {}
        confusion_payload: Dict[str, Dict[str, Any]] = {}
        benchmark_phase_start = time.perf_counter()

        for model_spec in models:
            try:
                training_start = time.perf_counter()
                trained_model = self.trainer.train_model(model_spec, x_fit, y_fit)
                training_time = time.perf_counter() - training_start
                inference_start = time.perf_counter()
                detail = tester.test_model(model_spec.name, trained_model, x_train, y_train, x_test, y_test)
                inference_time = time.perf_counter() - inference_start
                metrics = detail["metrics"].copy()
                metrics.update(
                    {
                        "model": model_spec.name,
                        "training_time_sec": training_time,      # updated after full-data retrain
                        "benchmark_training_time_sec": training_time,  # 70% split only, preserved
                        "inference_time_sec": inference_time,
                        "inference_ms_per_row": (inference_time / max(len(x_test), 1)) * 1000,
                        "complexity_score": _count_model_complexity(trained_model.named_steps["model"]),
                        "transformed_feature_count": int(
                            trained_model.named_steps["preprocessor"].transform(x_train.iloc[:1]).shape[1]
                        ),
                    }
                )
                benchmark_rows.append(metrics)
                benchmark_rows_by_name[model_spec.name] = metrics
                details[model_spec.name] = detail
                trained_models[model_spec.name] = trained_model
                confusion_payload[model_spec.name] = {
                    "matrix": np.asarray(detail["confusion_matrix"]).tolist(),
                    "labels": ["Actual 0", "Actual 1"],
                    "predicted": ["Predicted 0", "Predicted 1"],
                }
                artifacts.save_model(model_spec.name, trained_model)
            except Exception as exc:
                skipped_models[model_spec.name] = str(exc)

        if not benchmark_rows:
            raise RuntimeError(f"All requested models failed to train. Reasons: {skipped_models}")

        holdout_summary = pd.DataFrame(benchmark_rows).sort_values(
            ["f1", "accuracy", "roc_auc"], ascending=[False, False, False]
        )
        cv_results = self.trainer.k_fold_cross_validate(
            [model for model in models if model.name in trained_models], x_data, y_data, n_splits=cv_folds
        )
        artifacts.save_dataframe("cross_validation_results.csv", cv_results)
        artifacts.save_json("confusion_matrices.json", confusion_payload)
        benchmark_phase_wall_clock_sec = time.perf_counter() - benchmark_phase_start

        best_model_name = holdout_summary.iloc[0]["model"] if not holdout_summary.empty else None
        shap_explanations: List[Dict[str, Any]] = []
        shap_phase_start = time.perf_counter()
        if best_model_name:
            try:
                shap_explanations = self._save_shap_artifacts(
                    artifacts, best_model_name, trained_models[best_model_name], x_train, x_test, rows=min(shap_rows, len(x_test))
                )
            except Exception:
                shap_explanations = []
        shap_phase_wall_clock_sec = time.perf_counter() - shap_phase_start

        # ── Retrain every benchmarked model on the FULL dataset and save ──────
        print("\nRetraining the best model on full dataset for deployment...")
        full_data_models: Dict[str, Pipeline] = {}
        full_retrain_phase_start = time.perf_counter()
        if best_model_name and best_model_name in trained_models:
            try:
                full_start = time.perf_counter()
                full_model = self.trainer.retrain_from_benchmark(trained_models[best_model_name], x_data, y_data)
                full_time = time.perf_counter() - full_start
                full_data_models[best_model_name] = full_model
                if best_model_name in benchmark_rows_by_name:
                    row = benchmark_rows_by_name[best_model_name]
                    row["full_data_training_time_sec"] = full_time
                    row["training_time_sec"] = row["benchmark_training_time_sec"] + full_time
                print(f"  [{best_model_name}] full-data train: {full_time:.1f}s")
            except Exception as exc:
                print(f"  WARNING: Could not retrain {best_model_name} on full data: {exc}")
        full_retrain_phase_wall_clock_sec = time.perf_counter() - full_retrain_phase_start

        # Save the best-performing model as the deployment model
        deployment_model_name = best_model_name
        artifact_finalize_start = time.perf_counter()
        if deployment_model_name and deployment_model_name in full_data_models:
            deployment_path = artifacts.models_dir / "deployment_model.joblib"
            joblib.dump(full_data_models[deployment_model_name], deployment_path, compress=("xz", 3))
            print(f"  Deployment model saved: {deployment_model_name} -> deployment_model.joblib")
        elif trained_models:
            # Fallback: use the best available full-data model (or benchmark model if full-data failed)
            fallback_name = next(iter(full_data_models or trained_models))
            deployment_model_name = fallback_name
            fallback_model = (full_data_models or trained_models)[fallback_name]
            deployment_path = artifacts.models_dir / "deployment_model.joblib"
            joblib.dump(fallback_model, deployment_path, compress=("xz", 3))
            print(f"  Deployment model saved (fallback): {fallback_name} -> deployment_model.joblib")

        segmentation = self._save_segmentation_artifacts(artifacts, x_data)
        holdout_summary = pd.DataFrame(benchmark_rows).sort_values(
            ["f1", "accuracy", "roc_auc"], ascending=[False, False, False]
        )
        artifacts.save_dataframe("holdout_summary.csv", holdout_summary)
        artifact_finalize_wall_clock_sec = time.perf_counter() - artifact_finalize_start
        metadata = {
            "data_path": str(Path(data_path).resolve()),
            "dataset_kind": dataset_kind,
            "remove_leakage_features": remove_leakage_features,
            "feature_preset": resolved_preset,
            "balance_strategy": balance_strategy,
            "evaluation_mode": "Honest Prediction" if resolved_preset == "honest" else "High-Score Benchmark",
            "python_version": sys.version.split()[0],
            "train_rows": int(len(x_train)),
            "test_rows": int(len(x_test)),
            "total_rows": int(len(x_data)),
            "train_ratio": round(1.0 - self.trainer.test_size, 4),
            "test_ratio": round(self.trainer.test_size, 4),
            "cross_validation_folds": cv_folds,
            "best_model": best_model_name,
            "deployment_model": deployment_model_name,
            "trained_models": list(trained_models.keys()),
            "full_data_models": list(full_data_models.keys()),
            "skipped_models": skipped_models,
            "shap_explanations": shap_explanations,
            "segmentation_summary_rows": segmentation["summary"].to_dict(orient="records"),
            "tensorflow_version": None,
            "benchmark_phase_wall_clock_sec": benchmark_phase_wall_clock_sec,
            "shap_phase_wall_clock_sec": shap_phase_wall_clock_sec,
            "full_retrain_phase_wall_clock_sec": full_retrain_phase_wall_clock_sec,
            "artifact_finalize_wall_clock_sec": artifact_finalize_wall_clock_sec,
            "total_pipeline_wall_clock_sec": time.perf_counter() - pipeline_start,
        }
        metadata["pipeline_wall_clock_note"] = (
            "Total pipeline wall clock includes the benchmark holdout fits, cross-validation, "
            "SHAP generation, best-model full-data retraining for the deployment artifact, and report creation. "
            "Per-model training_time_sec is the sum of benchmark_training_time_sec and "
            "full_data_training_time_sec when a full-data retrain was performed."
        )
        try:
            metadata["tensorflow_version"] = importlib.import_module("tensorflow").__version__
        except ImportError:
            pass
        artifacts.save_json("metadata.json", metadata)
        artifacts.save_json(
            "prediction_schema.json",
            self._build_prediction_schema(prediction_inputs),
        )
        prediction_inputs.head(500).to_csv(artifacts.reports_dir / "prediction_examples.csv", index=False)
        return {"holdout_summary": holdout_summary, "cross_validation_results": cv_results, "metadata": metadata}

    def _build_prediction_schema(self, prediction_inputs: pd.DataFrame) -> Dict[str, Any]:
        schema: Dict[str, Any] = {"columns": []}
        categorical_columns = list(prediction_inputs.select_dtypes(include=["object", "category"]).columns)
        numeric_columns = [column for column in prediction_inputs.columns if column not in categorical_columns]
        for column in categorical_columns:
            series = prediction_inputs[column].astype(str).fillna("Unknown")
            mode = series.mode(dropna=True)
            schema["columns"].append(
                {
                    "name": column,
                    "type": "categorical",
                    "default": str(mode.iloc[0]) if not mode.empty else str(series.iloc[0]),
                    "options": sorted(series.unique().tolist()),
                }
            )
        for column in numeric_columns:
            series = pd.to_numeric(prediction_inputs[column], errors="coerce").fillna(0)
            schema["columns"].append(
                {
                    "name": column,
                    "type": "numeric",
                    "default": _safe_float(series.median()),
                    "min": _safe_float(series.min()),
                    "max": _safe_float(series.max()),
                    "step": 1.0 if pd.api.types.is_integer_dtype(series) else 0.1,
                }
            )
        return schema

    def _save_shap_artifacts(self, artifacts: TrainingArtifacts, model_name: str, model: Pipeline, x_train: pd.DataFrame, x_test: pd.DataFrame, rows: int):
        import matplotlib.pyplot as plt

        sample = x_test.sample(rows, random_state=self.random_state)
        analyzer = SHAPAnalyzer(random_state=self.random_state)
        shap_values = analyzer.explain(model, x_train, sample)
        summary_figure = analyzer.summary_plot(shap_values)
        summary_figure.savefig(artifacts.plots_dir / f"{_slugify(model_name)}_shap_summary.png", dpi=180)
        plt.close(summary_figure)
        values = np.asarray(shap_values.values)
        feature_names = list(shap_values.feature_names)
        mean_strength = np.abs(values).mean(axis=0)
        top_indices = np.argsort(mean_strength)[::-1][:3]
        explanations: List[Dict[str, Any]] = []
        for index in top_indices:
            feature_name = feature_names[index]
            feature_values = np.asarray(shap_values.data[:, index], dtype=float)
            shap_column = values[:, index]
            correlation = float(np.corrcoef(feature_values, shap_column)[0, 1]) if len(feature_values) > 1 else 0.0
            explanations.append(
                {
                    "feature": feature_name,
                    "mean_abs_shap": float(mean_strength[index]),
                    "correlation_with_risk": correlation,
                    "explanation": "Higher values tend to increase cancellation risk." if correlation >= 0 else "Higher values tend to reduce cancellation risk.",
                }
            )
        return explanations

    @staticmethod
    def _segment_display_names(summary: pd.DataFrame) -> dict[int, str]:
        if summary.empty or "segment" not in summary.columns:
            return {}
        working = summary.copy()
        for column in [
            "lead_time",
            "average_price",
            "number_of_total_nights",
            "number_of_children_and_adults",
            "special_requests",
            "cancellation_ratio",
        ]:
            if column in working.columns:
                working[column] = pd.to_numeric(working[column], errors="coerce").fillna(0)

        names: dict[int, str] = {}
        if "cancellation_ratio" in working.columns:
            risk_segment = int(working.sort_values("cancellation_ratio", ascending=False).iloc[0]["segment"])
            names[risk_segment] = "High-Risk Returners"
        if "lead_time" in working.columns:
            planner_segment = int(working.sort_values("lead_time", ascending=False).iloc[0]["segment"])
            names.setdefault(planner_segment, "Advance Planners")
        premium_features = [column for column in ("average_price", "special_requests") if column in working.columns]
        if premium_features:
            premium_scores = working[premium_features].mean(axis=1)
            premium_segment = int(working.iloc[int(premium_scores.idxmax())]["segment"])
            names.setdefault(premium_segment, "Premium Experience Guests")

        for row in working.itertuples(index=False):
            segment_id = int(getattr(row, "segment"))
            if segment_id in names:
                continue
            lead_time_value = float(getattr(row, "lead_time", 0.0))
            booking_value = float(getattr(row, "average_price", 0.0))
            if lead_time_value <= working["lead_time"].median() and booking_value <= working["average_price"].median():
                names[segment_id] = "Quick-Book Value Guests"
            else:
                names[segment_id] = "Steady Leisure Guests"
        return names

    def _save_segmentation_artifacts(self, artifacts: TrainingArtifacts, x_data: pd.DataFrame) -> Dict[str, Any]:
        import matplotlib.pyplot as plt

        segmenter = KMeansSegmenter(random_state=self.random_state, n_clusters=4)
        segmentation = segmenter.fit(x_data)
        segment_names = self._segment_display_names(segmentation["summary"])
        if not segmentation["summary"].empty:
            segmentation["summary"] = segmentation["summary"].copy()
            segmentation["summary"]["segment_name"] = segmentation["summary"]["segment"].map(segment_names)
        if not segmentation["projection"].empty:
            segmentation["projection"] = segmentation["projection"].copy()
            segmentation["projection"]["segment_name"] = segmentation["projection"]["segment"].map(segment_names)
        segmentation["summary"].to_csv(artifacts.reports_dir / "guest_segments.csv", index=False)
        projection = segmentation["projection"]
        figure, axis = plt.subplots(figsize=(8, 6))
        scatter = axis.scatter(
            projection["pc1"], projection["pc2"], c=projection["segment"], cmap="viridis", alpha=0.65, s=20
        )
        axis.set_title("Guest Segmentation with K-Means")
        axis.set_xlabel("Principal Component 1")
        axis.set_ylabel("Principal Component 2")
        figure.colorbar(scatter, ax=axis, label="Segment")
        figure.tight_layout()
        figure.savefig(artifacts.plots_dir / "guest_segmentation.png", dpi=180)
        plt.close(figure)
        return segmentation


## `hotel_app/ml/validation.py`


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable

import pandas as pd

from .models import BaseHotelModel
from .training import ModelTrainer


@dataclass
class ValidationRunner:
    trainer: ModelTrainer

    def run(
        self,
        model_specs: Iterable[BaseHotelModel],
        x_data: pd.DataFrame,
        y_data: pd.Series,
        n_splits: int = 5,
    ) -> pd.DataFrame:
        return self.trainer.k_fold_cross_validate(model_specs, x_data, y_data, n_splits=n_splits)


## `hotel_app/ml/models/__init__.py`


In [ ]:
from typing import Dict, Type

from hotel_app.ml.models.base import BaseHotelModel
from hotel_app.ml.models.logistic import LogisticRegressionModel
from hotel_app.ml.models.knn import KNNModel
from hotel_app.ml.models.decision_tree import DecisionTreeModel
from hotel_app.ml.models.svm import SVMModel
from hotel_app.ml.models.random_forest import RandomForestModel
from hotel_app.ml.models.xgboost_model import XGBoostModel
from hotel_app.ml.models.ann import ANNModel
from hotel_app.ml.models.lstm import LSTMModel
from hotel_app.ml.models.rnn import RNNModel

MODEL_REGISTRY: Dict[str, Type[BaseHotelModel]] = {
    model.name: model
    for model in (
        LogisticRegressionModel,
        KNNModel,
        DecisionTreeModel,
        SVMModel,
        RandomForestModel,
        XGBoostModel,
        ANNModel,
        LSTMModel,
        RNNModel,
    )
}

__all__ = [
    "BaseHotelModel",
    "LogisticRegressionModel",
    "KNNModel",
    "DecisionTreeModel",
    "SVMModel",
    "RandomForestModel",
    "XGBoostModel",
    "ANNModel",
    "LSTMModel",
    "RNNModel",
    "MODEL_REGISTRY"
]


## `hotel_app/ml/models/base.py`


In [ ]:
from __future__ import annotations

import inspect
from typing import Any

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight


class BaseHotelModel:
    """Base interface for every project model class.

    Each concrete model class is responsible for returning a configured
    estimator or search object through ``get_estimator()``. The shared
    pipeline then prepends the project preprocessor so every model is
    trained on the same transformed feature space.
    """
    name = "Base Model"

    def get_estimator(self) -> Any:
        raise NotImplementedError

    def build_pipeline(self, preprocessor: ColumnTransformer) -> Pipeline:
        return Pipeline(steps=[("preprocessor", clone(preprocessor)), ("model", self.get_estimator())])


class BalancedClassifierWrapper(BaseEstimator, ClassifierMixin):
    """Apply consistent balancing for models that need wrapper support.

    Some estimators support ``sample_weight`` natively, while others need
    simple oversampling to avoid the majority class dominating training.
    This wrapper keeps that logic in one place so the individual model
    classes stay small and readable.
    """
    def __init__(self, estimator: Any, strategy: str = "sample_weight", random_state: int = 42) -> None:
        self.estimator = estimator
        self.strategy = strategy
        self.random_state = random_state

    def fit(self, x_data: Any, y_data: Any) -> "BalancedClassifierWrapper":
        y_array = np.asarray(y_data)
        self.classes_ = np.unique(y_array)
        x_fit = x_data
        y_fit = y_array
        if self.strategy in {"oversample", "hybrid"}:
            x_fit, y_fit = self._oversample(x_data, y_array)
        fit_kwargs = {}
        supports_sample_weight = "sample_weight" in inspect.signature(self.estimator.fit).parameters
        if self.strategy in {"sample_weight", "hybrid"} and supports_sample_weight:
            fit_kwargs["sample_weight"] = compute_sample_weight(class_weight="balanced", y=np.asarray(y_fit))
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(x_fit, y_fit, **fit_kwargs)
        if hasattr(self.estimator_, "n_features_in_"):
            self.n_features_in_ = self.estimator_.n_features_in_
        return self

    def _oversample(self, x_data: Any, y_data: np.ndarray) -> tuple[Any, np.ndarray]:
        labels, counts = np.unique(y_data, return_counts=True)
        if len(labels) < 2 or counts.min() == counts.max():
            return x_data, y_data
        rng = np.random.default_rng(self.random_state)
        target = int(counts.max())
        sampled_indices = []
        for label in labels:
            label_indices = np.flatnonzero(y_data == label)
            extra = rng.choice(label_indices, size=target, replace=True)
            sampled_indices.append(extra)
        combined = np.concatenate(sampled_indices)
        rng.shuffle(combined)
        if isinstance(x_data, pd.DataFrame):
            return x_data.iloc[combined].reset_index(drop=True), y_data[combined]
        if isinstance(x_data, pd.Series):
            return x_data.iloc[combined].reset_index(drop=True), y_data[combined]
        array = np.asarray(x_data)
        return array[combined], y_data[combined]

    def predict(self, x_data: Any) -> Any:
        return self.estimator_.predict(x_data)

    def predict_proba(self, x_data: Any) -> Any:
        return self.estimator_.predict_proba(x_data)

    def decision_function(self, x_data: Any) -> Any:
        return self.estimator_.decision_function(x_data)


class SubsampledEstimatorWrapper(BaseEstimator, ClassifierMixin):
    """Fit expensive estimators on a stratified subset when needed.

    This is mainly useful for algorithms such as an RBF SVM whose training
    cost can explode on the full hotel-booking table. The wrapper keeps the
    estimator honest while bounding runtime and memory use.
    """

    def __init__(self, estimator: Any, max_samples: int = 20000, random_state: int = 42) -> None:
        self.estimator = estimator
        self.max_samples = max_samples
        self.random_state = random_state

    def fit(self, x_data: Any, y_data: Any) -> "SubsampledEstimatorWrapper":
        y_array = np.asarray(y_data)
        self.classes_ = np.unique(y_array)
        x_fit = x_data
        y_fit = y_array
        if self.max_samples and len(y_array) > self.max_samples:
            splitter = StratifiedShuffleSplit(
                n_splits=1,
                train_size=self.max_samples,
                random_state=self.random_state,
            )
            sample_index, _ = next(splitter.split(np.zeros(len(y_array)), y_array))
            if isinstance(x_data, pd.DataFrame):
                x_fit = x_data.iloc[sample_index]
            elif isinstance(x_data, pd.Series):
                x_fit = x_data.iloc[sample_index]
            else:
                x_fit = np.asarray(x_data)[sample_index]
            y_fit = y_array[sample_index]
        self.estimator_ = clone(self.estimator)
        self.estimator_.fit(x_fit, y_fit)
        if hasattr(self.estimator_, "n_features_in_"):
            self.n_features_in_ = self.estimator_.n_features_in_
        return self

    def predict(self, x_data: Any) -> Any:
        return self.estimator_.predict(x_data)

    def predict_proba(self, x_data: Any) -> Any:
        return self.estimator_.predict_proba(x_data)

    def decision_function(self, x_data: Any) -> Any:
        return self.estimator_.decision_function(x_data)


## `hotel_app/ml/models/logistic.py`


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from hotel_app.ml.models.base import BaseHotelModel


class LogisticRegressionModel(BaseHotelModel):
    """Tuned logistic baseline with native sigmoid probabilities.

    Doctor-facing notes:
    - estimator: ``LogisticRegression``
    - probability path: native ``predict_proba`` from logistic sigmoid
    - balancing: ``class_weight='balanced'``
    - tuning: ``GridSearchCV`` over the regularization strength ``C``
    """
    name = "Logistic Regression"

    def get_estimator(self) -> GridSearchCV:
        base_estimator = LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            solver="lbfgs",
            random_state=42,
        )
        return GridSearchCV(
            estimator=base_estimator,
            param_grid={
                "C": [0.35, 0.75, 1.25, 2.0],
            },
            scoring="accuracy",
            cv=3,
            n_jobs=-1,
            refit=True,
        )


## `hotel_app/ml/models/knn.py`


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from hotel_app.ml.models.base import BaseHotelModel, BalancedClassifierWrapper


class KNNModel(BaseHotelModel):
    """Tuned K-nearest neighbors classifier on scaled features.

    Doctor-facing notes:
    - estimator: ``KNeighborsClassifier``
    - probability path: native neighbor vote probabilities
    - balancing: oversampling wrapper before fit
    - tuning: ``GridSearchCV`` over neighbors, distance weighting, and metric
    """
    name = "KNN"

    def get_estimator(self) -> GridSearchCV:
        base_estimator = BalancedClassifierWrapper(
            KNeighborsClassifier(),
            strategy="oversample",
            random_state=42,
        )
        return GridSearchCV(
            estimator=base_estimator,
            param_grid={
                "estimator__n_neighbors": [15, 21, 27],
                "estimator__weights": ["distance", "uniform"],
                "estimator__p": [1, 2],
                "estimator__leaf_size": [20, 30],
            },
            scoring="accuracy",
            cv=3,
            n_jobs=-1,
            refit=True,
        )


## `hotel_app/ml/models/decision_tree.py`


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from hotel_app.ml.models.base import BaseHotelModel


class DecisionTreeModel(BaseHotelModel):
    """Single-tree classifier tuned for an interpretable baseline.

    Doctor-facing notes:
    - estimator: ``DecisionTreeClassifier``
    - probability path: native leaf-class probabilities
    - balancing: ``class_weight='balanced'``
    - tuning: ``GridSearchCV`` over depth and split/leaf controls
    """
    name = "Decision Tree"

    def get_estimator(self) -> GridSearchCV:
        base_estimator = DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42,
        )
        return GridSearchCV(
            estimator=base_estimator,
            param_grid={
                "criterion": ["gini", "entropy"],
                "max_depth": [8, 12, 16],
                "min_samples_split": [2, 8],
                "min_samples_leaf": [4, 8, 12],
            },
            scoring="accuracy",
            cv=3,
            n_jobs=-1,
            refit=True,
        )


## `hotel_app/ml/models/random_forest.py`


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from hotel_app.ml.models.base import BaseHotelModel


class RandomForestModel(BaseHotelModel):
    """Random forest with search-based tuning for the strongest tabular baseline.

    Doctor-facing notes:
    - estimator: ``RandomForestClassifier``
    - probability path: averaged tree probabilities
    - balancing: ``class_weight='balanced_subsample'``
    - tuning: ``RandomizedSearchCV`` over tree count, depth, and split controls
    """
    name = "Random Forest"
    khaled_random_forest_search_space = {
        "n_estimators": [120, 180, 240, 320],
        "max_depth": [12, 18, 24, None],
        "min_samples_split": [2, 4, 8],
        "min_samples_leaf": [1, 2, 4],
        "max_features": [0.3, 0.35, 0.45, "sqrt"],
        "bootstrap": [True, False],
    }

    def get_estimator(self) -> RandomizedSearchCV:
        base_estimator = RandomForestClassifier(
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1,
        )
        return RandomizedSearchCV(
            estimator=base_estimator,
            param_distributions=self.khaled_random_forest_search_space,
            n_iter=8,
            scoring="accuracy",
            cv=3,
            random_state=42,
            n_jobs=-1,
            refit=True,
        )


## `hotel_app/ml/models/svm.py`


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from hotel_app.ml.models.base import BaseHotelModel, SubsampledEstimatorWrapper


class SVMModel(BaseHotelModel):
    """RBF-kernel support vector machine with bounded training size.

    Doctor-facing notes:
    - estimator: ``SVC(kernel='rbf', probability=True)`` on a stratified subset
    - probability path: native ``predict_proba`` from the fitted SVC
    - balancing: ``class_weight='balanced'``
    - tuning: ``GridSearchCV`` over ``C`` and ``gamma``
    """
    name = "SVM"

    def get_estimator(self) -> GridSearchCV:
        base_estimator = SubsampledEstimatorWrapper(
            SVC(
                kernel="rbf",
                probability=True,
                class_weight="balanced",
                random_state=42,
            ),
            max_samples=8000,
            random_state=42,
        )
        return GridSearchCV(
            estimator=base_estimator,
            param_grid={
                "estimator__C": [2.0, 5.0],
                "estimator__gamma": ["scale", 0.01],
            },
            scoring="accuracy",
            cv=3,
            n_jobs=1,
            refit=True,
        )


## `hotel_app/ml/models/ann.py`


In [ ]:
from typing import Any
import importlib
from sklearn.neural_network import MLPClassifier
from hotel_app.ml.models.base import BaseHotelModel, BalancedClassifierWrapper
from hotel_app.ml.deep import KerasTabularClassifier


class ANNModel(BaseHotelModel):
    """Feed-forward neural network for dense tabular classification.

    Doctor-facing notes:
    - estimator: TensorFlow ``KerasTabularClassifier`` in ANN mode, with an
      ``MLPClassifier`` fallback when TensorFlow is unavailable
    - probability path: final dense layer uses ``sigmoid`` activation
    - balancing: class-weighted TensorFlow fit or oversampling fallback
    - tuning: architecture and epochs are controlled from the training runner
    """
    name = "ANN"

    def __init__(self, epochs: int = 120, batch_size: int = 192) -> None:
        self.khaled_ann_epochs = epochs
        self.khaled_ann_batch_size = batch_size

    def get_estimator(self) -> Any:
        try:
            importlib.import_module("tensorflow")
            return KerasTabularClassifier(
                model_type="ann",
                epochs=self.khaled_ann_epochs,
                batch_size=self.khaled_ann_batch_size,
            )
        except ImportError:
            return BalancedClassifierWrapper(
                MLPClassifier(
                    hidden_layer_sizes=(128, 64, 32),
                    activation="relu",
                    learning_rate_init=0.001,
                    batch_size=min(self.khaled_ann_batch_size, 256),
                    max_iter=self.khaled_ann_epochs,
                    early_stopping=True,
                    validation_fraction=0.1,
                    random_state=42,
                ),
                strategy="oversample",
                random_state=42,
            )


## `hotel_app/ml/models/rnn.py`


In [ ]:
from hotel_app.ml.models.base import BaseHotelModel
from hotel_app.ml.deep import KerasTabularClassifier


class RNNModel(BaseHotelModel):
    """Simple recurrent neural network for reshaped tabular sequences.

    Doctor-facing notes:
    - estimator: TensorFlow ``KerasTabularClassifier`` in RNN mode
    - probability path: final dense layer uses ``sigmoid`` activation
    - balancing: class-weighted TensorFlow fit
    - tuning: epochs and batch size are exposed through the model class
    """
    name = "RNN"

    def __init__(self, epochs: int = 45, batch_size: int = 192) -> None:
        self.khaled_rnn_epochs = epochs
        self.khaled_rnn_batch_size = batch_size

    def get_estimator(self) -> KerasTabularClassifier:
        return KerasTabularClassifier(
            model_type="rnn",
            epochs=self.khaled_rnn_epochs,
            batch_size=self.khaled_rnn_batch_size,
        )


## `hotel_app/ml/models/lstm.py`


In [ ]:
from hotel_app.ml.models.base import BaseHotelModel
from hotel_app.ml.deep import KerasTabularClassifier


class LSTMModel(BaseHotelModel):
    """LSTM network for the sequential deep-learning variant.

    Doctor-facing notes:
    - estimator: TensorFlow ``KerasTabularClassifier`` in LSTM mode
    - probability path: final dense layer uses ``sigmoid`` activation
    - balancing: class-weighted TensorFlow fit
    - tuning: epochs and batch size are exposed through the model class
    """
    name = "LSTM"

    def __init__(self, epochs: int = 55, batch_size: int = 192) -> None:
        self.khaled_lstm_epochs = epochs
        self.khaled_lstm_batch_size = batch_size

    def get_estimator(self) -> KerasTabularClassifier:
        return KerasTabularClassifier(
            model_type="lstm",
            epochs=self.khaled_lstm_epochs,
            batch_size=self.khaled_lstm_batch_size,
        )


## `hotel_app/ml/models/xgboost_model.py`


In [ ]:
from typing import Any
import importlib
from hotel_app.ml.models.base import BaseHotelModel


class XGBoostModel(BaseHotelModel):
    name = "XGBoost"

    def get_estimator(self) -> Any:
        try:
            xgboost = importlib.import_module("xgboost")
        except ImportError as exc:
            raise ImportError(
                "XGBoost is required for XGBoostModel. Install it with `pip install xgboost` "
                "or remove XGBoost from the selected models."
            ) from exc
        return xgboost.XGBClassifier(
            booster="gbtree",
            learning_rate=0.05,
            max_depth=6,
            n_estimators=320,
            min_child_weight=2,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=1.5,
            eval_metric="logloss",
            tree_method="hist",
            n_jobs=1,
            random_state=42,
        )
